 # **Cart Abandonment vs Purchase analysis**

This notebook provides a comprehensive analysis of user behavior within an e-commerce funnel, focusing on identifying key factors that influence cart abandonment versus successful purchases. The analysis proceeds through several structured stages:

*   **0 Import Libraries and Dataframe**: Essential libraries are loaded, and the raw sales data is imported into a pandas DataFrame for initial inspection.
*   **1 Preprocessing**: This section focuses on cleaning and preparing the data. It involves correcting inconsistencies across related columns, managing null values, and conducting an outlier analysis to ensure data quality and integrity.
*   **2 Feature engineering**: New features are created, and existing ones are transformed to enhance the predictive power of the models. This includes logarithmic transformations, categorization of user and session IDs, calculation of interaction ratios, and the development of historical user behavior features.
*   **3 Funnel-Based Logistic Regression (Chain Rule Decomposition)**: A series of Logistic Regression models are built to estimate the probability of user progression through three key stages of the sales funnel: 'Viewed Product to Added to Cart', 'Added to Cart to Started Checkout', and 'Started Checkout to Purchased'. This chain-rule approach provides stage-specific insights into conversion dynamics.
*   **4 Funnel-Based Random Forest (Chain Rule Decomposition)**: The funnel analysis is replicated using Random Forest classifiers to explore non-linear relationships and interaction effects between features. This section also includes cross-validation to assess model stability and generalization capabilities.
*   **5. Business Recommendations for Versace — Executive Summary**: Based on the findings from both the Logistic Regression and Random Forest models, actionable business recommendations are provided to help Versace optimize its e-commerce strategy, reduce abandonment, and increase conversion rates.

## 0 Import Libraries and Dataframe

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import matplotlib.dates as mdates
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, accuracy_score
from sklearn.model_selection import train_test_split, GroupKFold, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

In [ ]:
df = pd.read_csv("IMA2026_Versace_Cart_Abandonment.csv",
                 sep=',',
                 on_bad_lines='skip',
                 dtype={"user_pseudo_id": "object", "user_id": "object"},
                )
df.head()

## 1 Preprocessing

In this section, we will systematically prepare and clean the dataset for robust analysis. The primary goal is to address data quality issues, ensuring that the information is consistent and accurate across all variables. This process is crucial as it forms the foundation for reliable feature engineering and model development.

Specifically, we will examine the variables in the dataset to identify inconsistencies and understand the reasons behind many null values in various columns. Initially, some inconsistencies were noted manually (e.g., by exporting the file to Excel) and subsequently verified and logically corrected step by step through code. This systematic approach allows us to address issues related to missing values properly, using the code just to correct those parts that are logically inconsistent as can be observed from the main insights for each column. Furthermore, in subsection 1.6, a detailed outlier analysis will be conducted to identify and evaluate extreme values in the data.

In [ ]:
df.info()

As observed from `df.info()`, many columns contain a significant number of null values. This is an expected characteristic of this dataset, as it represents a user funnel. Users do not always proceed through every stage of the funnel (e.g., from viewing a product to adding it to a cart, then to checkout, and finally to purchase). Consequently, data related to later stages will naturally be null for sessions that did not reach those stages, reflecting a non-linear progression of user behavior.

### 1.1 Data Cleaning and Preprocessing

Here, we will check for duplicate rows in the dataset. Based on our findings, the dataset contains no duplicate rows.

In [ ]:
# Convert 'user_id' to string to handle mixed types (as indicated by DtypeWarning) and 'event_date' to datetime.
df['event_date'] = pd.to_datetime(df['event_date'])

print("Data types updated for and 'event_date'.")

# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_rows}\n")

# If duplicates exist, you might want to remove them:
# if duplicate_rows > 0:
#     df.drop_duplicates(inplace=True)
#     print(f"{duplicate_rows} duplicate rows removed.")

# Display updated DataFrame information

### 1.2 Correct for inconsistencies
In this section, we aim to ensure the homogeneity of the dataset among columns that, while correlated, must provide the same information. Consistency checks were performed by referencing the data dictionary and through specific value analysis, rather than relying on generic algorithmic checks. This approach allows us to identify and correct logical inconsistencies arising from the intrinsic meaning of the variables.

#### 1.2.1 Consistency given by `drop_off_stage`.
In this subsection we aim at correct inconsistencies between `drop_off_stage` (which is our reference variables, since it shows no missing values, suggesting its very high reliability in terms of information provided) and the dummy variables called `purchased`, `viewed_product`, `added_to_cart`, `started_checkout`.

As shown below, `drop_off_stage` contains 5 different categories: `no_product_interaction`,`product_abandonment`,
`cart_abandonment`, `converted` and `checkout_abandonment`. The idea is the following: if a user in a given section shows that he/she abandoned the cart, than the corresponding dummy variables called `viewed_product` and `added_to_cart` should have their correposnding entry equal to 1, while `purchased` and `started_checkout` should have their corresponding entry equal to 0. Similar reasoning apply to the other `drop_off_stage` categories.

This approach allows us to retain some entries in the dataset that would have otherwise dropped out because inconsistent across rows.



In [ ]:
df["drop_off_stage"].unique()

In [ ]:
# === 2) Define the 'drop_off_stage' -> correct funnel state mapping ===
stage_map = {
    "no_product_interaction": {
        "viewed_product": 0,
        "added_to_cart": 0,
        "started_checkout": 0,
        "purchased": 0,
    },
    "product_abandonment": {
        "viewed_product": 1,
        "added_to_cart": 0,
        "started_checkout": 0,
        "purchased": 0,
    },
    "cart_abandonment": {
        "viewed_product": 1,
        "added_to_cart": 1,
        "started_checkout": 0,
        "purchased": 0,
    },
    "checkout_abandonment": {
        "viewed_product": 1,
        "added_to_cart": 1,
        "started_checkout": 1,
        "purchased": 0,
    },
    "converted": {
        "viewed_product": 1,
        "added_to_cart": 1,
        "started_checkout": 1,
        "purchased": 1,
    },
}

funnel_cols = ["viewed_product", "added_to_cart", "started_checkout", "purchased"]

# === 3) Optional backup of original columns ===
for col in funnel_cols:
    df[f"{col}_original"] = df[col]

# === 4) Check for unexpected 'drop_off_stage' values ===
unknown_stages = set(df["drop_off_stage"].dropna().unique()) - set(stage_map.keys())
if unknown_stages:
    raise ValueError(f"Unexpected values found in drop_off_stage: {unknown_stages}")

# === 5) Correct funnel columns using drop_off_stage as base ===
for stage, values in stage_map.items():
    mask = df["drop_off_stage"] == stage
    for col, corrected_value in values.items():
        df.loc[mask, col] = corrected_value

# === 6) Report of corrections made ===
correction_report = {}
for col in funnel_cols:
    changed = (df[col] != df[f"{col}_original"]).sum()
    correction_report[col] = int(changed)
df = df.drop(columns=["viewed_product_original", "added_to_cart_original", "started_checkout_original", "purchased_original"])

print("Number of corrected values per column:")
for col, n in correction_report.items():
    print(f"- {col}: {n}")
print(f"New dataset shape: {df.shape}")

This homogenization procedure resulted in changes in the corresponding entries for several dummy variables. Specifically, we observed 162 corrections in `viewed_product`, 240 in `added_to_cart`, and 43 in `started_checkout`. The `purchased` column had only 1 change, indicating a high initial consistency.

#### 1.2.2 Consistency between Items and Unique items.

In this subsection, we ensure consistency between `items_viewed` and `unique_items_viewed`, as well as `items_added_to_cart` and `unique_items_added_to_cart`. Specifically, we verify that the count in each `unique_` column does not exceed its corresponding non-unique counterpart. Should an inconsistency be found (e.g., `unique_items_viewed` > `items_viewed`), the affected record will be removed from the dataset.

In [ ]:
cols = df.columns.tolist()

# Automatically find all columns starting with "unique_"
unique_cols = [c for c in cols if c.startswith("unique_")]

invalid_masks = []
checked_pairs = []

for ucol in unique_cols:
    base_col = ucol.replace("unique_", "", 1)

    # Check if the base column exists
    if base_col in df.columns:
        checked_pairs.append((ucol, base_col))

        # Inconsistent row if unique > total
        mask = df[ucol] > df[base_col]
        invalid_masks.append(mask)

# Combine all conditions
if invalid_masks:
    invalid_rows_mask = pd.concat(invalid_masks, axis=1).any(axis=1)
else:
    invalid_rows_mask = pd.Series(False, index=df.index)

# Report
print("Checked pairs:")
for ucol, base_col in checked_pairs:
    n_invalid = (df[ucol] > df[base_col]).sum()
    print(f"- {ucol} vs {base_col}: {n_invalid} inconsistent rows")

print(f"\nTotal rows to remove: {invalid_rows_mask.sum()}")

# Remove inconsistent rows
df = df.loc[~invalid_rows_mask].copy()

print(f"New dataset shape: {df.shape}")

The results confirm that the code did not detect any inconsistencies within the two pairs of columns, meaning no rows were removed from the dataset.

#### 1.2.3 Consistency between Cart Value and Avg_viewed_price with drop_off_stage

This subsection is based on the analysis of `cart value` and `avg_viewed_price` with the variable `drop_off_stage`. The first 2 variables show several null values, which are not an issue "per-se", but they may arise concerns if those values are null when a user either bought a product or at least dropped off when he/she put something in the cart. Therefore, we decided to eliminate those entries.

Specifically, for sessions with `no_product_interaction` or `product_abandonment`, `cart_value` is set to 0. For `product_abandonment`, rows are removed if `avg_viewed_price` is missing or zero. For `cart_abandonment`, `checkout_abandonment`, and `converted` stages, rows are removed if `cart_value` or `avg_viewed_price` are missing or zero.

In [ ]:
# Convert to numeric: empty strings or invalid values become NaN
df["cart_value"] = pd.to_numeric(df["cart_value"], errors="coerce")
df["avg_viewed_price"] = pd.to_numeric(df["avg_viewed_price"], errors="coerce")

# Initial shape
print("Initial shape:", df.shape)

# Masks for drop_off_stage
mask_no_product = df["drop_off_stage"] == "no_product_interaction"
mask_product_abandonment = df["drop_off_stage"] == "product_abandonment"
mask_later_stages = df["drop_off_stage"].isin([
    "cart_abandonment",
    "checkout_abandonment",
    "converted"
])

# 1) no_product_interaction:
# if keeping the row, set both to 0
df.loc[mask_no_product, "cart_value"] = 0
df.loc[mask_no_product, "avg_viewed_price"] = 0

# 2) product_abandonment:
# cart_value consistent with stage -> 0
df.loc[mask_product_abandonment, "cart_value"] = 0

# drop if avg_viewed_price is empty or <= 0
drop_product_abandonment = (
    mask_product_abandonment &
    (
        df["avg_viewed_price"].isna() |
        (df["avg_viewed_price"] <= 0)
    )
)

# 3) cart_abandonment / checkout_abandonment / converted:
# drop if cart_value is empty or <= 0
# or if avg_viewed_price is empty or <= 0
drop_later_stages = (
    mask_later_stages &
    (
        df["cart_value"].isna() |
        (df["cart_value"] <= 0) |
        df["avg_viewed_price"].isna() |
        (df["avg_viewed_price"] <= 0)
    )
)

# Final mask of rows to remove
rows_to_drop = drop_product_abandonment | drop_later_stages

print("Rows to remove for product_abandonment:", int(drop_product_abandonment.sum()))
print("Rows to remove for cart/checkout/converted:", int(drop_later_stages.sum()))
print("Total rows to remove:", int(rows_to_drop.sum()))

# Remove inconsistent rows
df = df.loc[~rows_to_drop].copy()

# Final shape
print("Final shape:", df.shape)

Here we removed a total of 290 rows from the dataset that showed inconsistencies between the `drop_off_stage` and the values of `cart_value` or `avg_viewed_price`.

Specifically, 8 rows were removed for the `product_abandonment` stage (where `avg_viewed_price` was null or <= 0) and 282 rows for the later stages (`cart_abandonment`, `checkout_abandonment`, `converted`) where `cart_value` or `avg_viewed_price` were null or <= 0.

#### 1.2.4 Consistency in Time Variables

In this subsection, we analyze the various time-related columns to assess their redundancy and consistency. The dataset includes four main time event categories: `first_view`, `first_cart`, `first_checkout`, and `purchase`.

For each of these event categories, five variables were generated with the following suffixes: `_ts` (timestamp), `_datetime`, `_time`, `_date`, and `_hour`. Our analysis will proceed pairwise, comparing these related columns to identify redundancies. Based on these checks, we will determine which columns can be safely dropped without losing valuable information.

The first pair of groups of columns is given by those 4 ending with `_ts` and `_datetime` to see if they provide the same information.

In [ ]:
# Pairs to check
pairs = [
    ("first_view_ts", "first_view_datetime"),
    ("first_cart_ts", "first_cart_datetime"),
    ("first_checkout_ts", "first_checkout_datetime"),
    ("purchase_ts", "purchase_datetime"),
]

# 1) Safe conversion of columns
for col_ts, col_dt in pairs:
    # Convert _ts only if it's not already datetime
    if not pd.api.types.is_datetime64_any_dtype(df[col_ts]):
        df[col_ts] = pd.to_numeric(df[col_ts], errors="coerce")
        df[col_ts] = pd.to_datetime(df[col_ts], unit="us", errors="coerce")

    # Convert _datetime if it's not already datetime
    if not pd.api.types.is_datetime64_any_dtype(df[col_dt]):
        df[col_dt] = pd.to_datetime(df[col_dt], errors="coerce")

# 2) Consistency check between pairs
for col_ts, col_dt in pairs:
    same_info_mask = (
        (df[col_ts] == df[col_dt]) |
        (df[col_ts].isna() & df[col_dt].isna())
    )

    n_same = int(same_info_mask.sum())
    n_diff = int((~same_info_mask).sum())

    print(f"\nChecking {col_ts} vs {col_dt}")
    print(f"- rows containing the same information: {n_same}")
    print(f"- rows NOT containing the same information: {n_diff}")

    if n_diff > 0:
        print("Examples of inconsistent rows:")
        display(df.loc[~same_info_mask, [col_ts, col_dt]].head())

The code confirms that `_ts` and `date_time` contain the same information.

In the next cell, we test if the date contained in `_ts` is the same as the one contained in `_date`, for each of the 4 variables.

In [ ]:
date_checks = [
    ("first_view_ts", "first_view_date"),
    ("first_cart_ts", "first_cart_date"),
    ("first_checkout_ts", "first_checkout_date"),
    ("purchase_ts", "purchase_date"),
]

# Convert _date columns to datetime if not already
for col_ts, col_date in date_checks:
    if not pd.api.types.is_datetime64_any_dtype(df[col_ts]):
        df[col_ts] = pd.to_datetime(df[col_ts], errors="coerce")

    if not pd.api.types.is_datetime64_any_dtype(df[col_date]):
        df[col_date] = pd.to_datetime(df[col_date], errors="coerce")

# Check: the date extracted from _ts must match _date
for col_ts, col_date in date_checks:
    ts_date = df[col_ts].dt.date
    date_only = df[col_date].dt.date

    same_date_mask = (
        (ts_date == date_only) |
        (df[col_ts].isna() & df[col_date].isna())
    )

    n_same = int(same_date_mask.sum())
    n_diff = int((~same_date_mask).sum())

    print(f"\nChecking DATE {col_ts} vs {col_date}")
    print(f"- rows with the same date: {n_same}")
    print(f"- rows with different dates: {n_diff}")

    if n_diff > 0:
        print("Examples of inconsistent rows:")
        display(df.loc[~same_date_mask, [col_ts, col_date]].head())

The previous checks confirmed that the date components of `_ts` and `_date` columns are consistent.  

Next, we verified the consistency between the time components of `_ts` and `_time` columns. This check also revealed no inconsistencies across any of the pairs, indicating perfect compatibility.

In [ ]:
time_checks = [
    ("first_view_ts", "first_view_time"),
    ("first_cart_ts", "first_cart_time"),
    ("first_checkout_ts", "first_checkout_time"),
    ("purchase_ts", "purchase_time"),
]

for col_ts, col_time in time_checks:
    # _ts: if not already datetime, try converting it
    if not pd.api.types.is_datetime64_any_dtype(df[col_ts]):
        ts_numeric = pd.to_numeric(df[col_ts], errors="coerce")
        df[col_ts] = pd.to_datetime(ts_numeric, unit="us", errors="coerce")

    # _time: convert to datetime only to extract the hour
    time_parsed = pd.to_datetime(df[col_time].astype(str), errors="coerce")

    # Extract the time
    ts_time = df[col_ts].dt.strftime("%H:%M:%S.%f")
    time_only = time_parsed.dt.strftime("%H:%M:%S.%f")

    # Consistency: same time or both missing
    same_time_mask = (
        (ts_time == time_only) |
        (df[col_ts].isna() & time_parsed.isna())
    )

    n_same = int(same_time_mask.sum())
    n_diff = int((~same_time_mask).sum())

    print(f"\nChecking TIME {col_ts} vs {col_time}")
    print(f"- rows with the same time: {n_same}")
    print(f"- rows with different times: {n_diff}")

    if n_diff > 0:
        print("Examples of inconsistent rows:")
        display(df.loc[~same_time_mask, [col_ts, col_time]].head())

Finally, we chech if the time in the 4 variables ending with `_time` is consistent with the hour in the 4 columns ending with `_hour`.

In [ ]:
hour_checks = [
    ("first_view_time", "first_view_hour"),
    ("first_cart_time", "first_cart_hour"),
    ("first_checkout_time", "first_checkout_hour"),
    ("purchase_time", "purchase_hour"),
]

invalid_masks = []

for time_col, hour_col in hour_checks:
    # Convert _time to datetime only to extract the hour
    time_parsed = pd.to_datetime(df[time_col].astype(str), errors="coerce")

    # Convert _hour to numeric
    hour_numeric = pd.to_numeric(df[hour_col], errors="coerce")

    # Hour extracted from _time
    time_hour = time_parsed.dt.hour

    # Consistent if equal, or both missing
    valid_mask = (
        (time_hour == hour_numeric) |
        (time_parsed.isna() & hour_numeric.isna())
    )

    invalid_mask = ~valid_mask
    invalid_masks.append(invalid_mask)

    print(f"\nChecking {time_col} vs {hour_col}")
    print(f"- consistent rows: {int(valid_mask.sum())}")
    print(f"- inconsistent rows: {int(invalid_mask.sum())}")

# Rows to drop if inconsistent in at least one pair
rows_to_drop = pd.concat(invalid_masks, axis=1).any(axis=1)

print(f"\nTotal rows to remove: {int(rows_to_drop.sum())}")

df = df.loc[~rows_to_drop].copy()

print("New dataset shape:", df.shape)

The output confirms that the time components extracted from the `_time` columns are perfectly consistent with the `_hour` columns for all four event types (`first_view`, `first_cart`, `first_checkout`, `purchase`). There were no inconsistent rows detected, meaning no data was removed from the DataFrame during this check. This validates the integrity of these time-related features.

This last check shows that all the columns are consistent, which means that we are now allowed to remove those columns that do not provide further information, given the others.

In [ ]:
# Drop *_datetime columns and transform *_ts columns to time only

cols_to_drop = [
    "first_view_datetime",
    "first_cart_datetime",
    "first_checkout_datetime",
    "purchase_datetime",
    "first_view_ts",
    "first_cart_ts",
    "first_checkout_ts",
    "purchase_ts",
]

ts_cols = [
    "first_view_ts",
    "first_cart_ts",
    "first_checkout_ts",
    "purchase_ts",
]


# Ensure ts columns are datetime
for col in ts_cols:
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Keep only the time from ts columns
for col in ts_cols:
    df[col] = df[col].dt.time

# Drop duplicate datetime columns
df = df.drop(columns=cols_to_drop)



# Final check
print(df[[
    "first_view_date", "first_view_time",
    "first_cart_date", "first_cart_time",
    "first_checkout_date", "first_checkout_time",
    "purchase_date", "purchase_time"
]].head())

print(df.shape)

#### 1.2.5 Consistency in the `time_to` columns

In this last subsection, we want to eliminate the records that have a negative value in one of the 3 columns starting with `time_`: `time_view_to_cart`, `time_cart_to_checkout` and `time_checkout_to_purchase`.

In [ ]:
time_cols = [
    "time_view_to_cart",
    "time_cart_to_checkout",
    "time_checkout_to_purchase"
]

# Convert to numeric: empty values remain NaN and will NOT be dropped
for col in time_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Initial shape:", df.shape)

# Rows to remove: at least one negative value
rows_to_drop = (df[time_cols] < 0).any(axis=1)

# Report per column
for col in time_cols:
    n_neg = int((df[col] < 0).sum())
    print(f"{col}: negative values = {n_neg}")

print(f"\nTotal rows to remove: {int(rows_to_drop.sum())}")

# Drop rows with at least one negative
df = df.loc[~rows_to_drop].copy()

print("Final shape:", df.shape)

#### 1.2.6 Consistency between `total_engagement_time` and `session_duration_sec`

In this subsection, we ensure that `total_engagement_time` (converted to seconds) is logically less than or equal to `session_duration_sec`. If `total_engagement_time` exceeds `session_duration_sec`, it indicates an inconsistency, and such records will be removed from the dataset.

In [ ]:
# Convert total_engagement_time from milliseconds to seconds
df['total_engagement_time_sec'] = df['total_engagement_time'] / 1000

print("Initial shape:", df.shape)

# Identify rows where total_engagement_time_sec is greater than session_duration_sec
inconsistent_engagement_mask = (
    (df['total_engagement_time_sec'] > df['session_duration_sec'])
)

# Report and remove inconsistent rows
num_inconsistent_rows = inconsistent_engagement_mask.sum()
print(f"Number of inconsistent rows where total_engagement_time_sec > session_duration_sec: {int(num_inconsistent_rows)}")

df = df.loc[~inconsistent_engagement_mask].copy()

# Drop the temporary column
df = df.drop(columns=['total_engagement_time_sec'])

print("Final shape:", df.shape)

In [ ]:
print(df['total_engagement_time'].describe())
print("\nSample values:")
print(df['total_engagement_time'].dropna().head(10).values)

At the end of this preprocessing section, we are left with 192934 rows and 38 columns, compared to the initial 212049 rows and 39 columns present in the intiial dataset.

### 1.3 Outlier Analysis

In [ ]:
# Focus on continuous numerical variables available at this stage
outlier_cols = [
    'items_viewed', 'unique_items_viewed', 'items_added_to_cart',
    'unique_items_added_to_cart', 'unique_categories_viewed',
    'cart_value', 'avg_viewed_price', 'total_engagement_time',
    'session_duration_sec', 'time_view_to_cart', 'time_cart_to_checkout'
]
outlier_cols = [c for c in outlier_cols if c in df.columns]

Q1 = df[outlier_cols].quantile(0.25)
Q3 = df[outlier_cols].quantile(0.75)
IQR = Q3 - Q1

outlier_mask = ((df[outlier_cols] < (Q1 - 3 * IQR)) |
                (df[outlier_cols] > (Q3 + 3 * IQR)))

print("Number of outliers per variable (IQR method, threshold = 3):")
print(outlier_mask.sum().sort_values(ascending=False))
print(f"\nRows with at least one outlier: {outlier_mask.any(axis=1).sum()}")
print(f"As % of dataset: {outlier_mask.any(axis=1).mean()*100:.1f}%")

In [ ]:
# The definition of outlier_cols and subsequent calculations have been moved to the cell above.

In [ ]:
# Boxplot of outlier variables to visualize their extent
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    flierprops=dict(marker='o', markersize=2,
                                    color='tomato', alpha=0.3))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Value')

# Hide unused subplots
for j in range(len(outlier_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outlier Distribution by Variable (IQR threshold = 3)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

#### Interpretation and Decision

The outlier analysis reveals that **16.4% of sessions contain at least one extreme
value** across the behavioral variables. The distribution of outliers is highly
uneven across variables:

- **`session_duration_sec` (18,433 outliers)** and **`total_engagement_time`
  (7,253)** show the highest outlier counts. This is expected in e-commerce
  behavioral data: a small subset of highly engaged users — such as users who
  leave a tab open for hours or repeatedly revisit the same page — naturally
  generates extreme session durations. These are genuine high-intent signals,
  not measurement errors.

- **`items_viewed`, `unique_items_viewed`, `items_added_to_cart`,
  `unique_items_added_to_cart`, `cart_value`** (5,000–8,000 outliers each)
  reflect power-user behavior: a minority of users view many products or build
  large carts. Again, these are informative signals that should be retained.

- **`avg_viewed_price` (6,629 outliers)** captures sessions focused on very
  high-end or very low-end items, which is structurally meaningful in a luxury
  brand context where price range varies widely across product categories.

- **`time_view_to_cart` (400)** and **`time_cart_to_checkout` (125)** show
  few outliers, suggesting that the timing variables are relatively well-behaved.

**Decision: outliers are retained in the dataset.** Removal would eliminate
16.4% of observations — a substantial loss — and would systematically discard
precisely the high-engagement users who are most interesting from a conversion
modeling perspective. Instead, the log-transformations applied in Section 2.1
(`log1p`) substantially compress the scale of all these variables, reducing
the influence of extreme values on model estimation without any data loss.
This is the standard approach in e-commerce behavioral analytics.

This chart shows that **total_engagement_time is non-null for the large majority of records across all drop-off stages**, with **missing values concentrated mainly in the earlier, less advanced stages of the funnel**.
In particular, the **only meaningful share of null values appears in no_product_interaction and, to a lesser extent, product_abandonment**. This is a reasonable pattern: for users who never reached the cart stage, having total_engagement_time equal to 0 or missing is expected, because they did not progress far enough in the journey to generate that type of measurable engagement.
By contrast, for **cart_abandonment, checkout_abandonment, and converted users, the metric is almost entirely populated**, which is consistent with the fact that these users moved deeper into the funnel and therefore produced trackable engagement.
Overall, this **does not suggest a data quality issue**. Instead, it reflects **normal user behavior across funnel stages**. For that reason, there is **no strong need to remove those records**, since the null or zero engagement time for users who did not reach the cart is logically coherent and should be considered part of the expected funnel structure.

### 1.4 Data variability check

In [ ]:
print("Checking for columns with no variability (all same entry) or high variability (all unique entries):\n")

# Get the number of rows in the DataFrame
num_rows = len(df)

constant_columns = []
unique_columns = []

for col in df.columns:
    num_unique = df[col].nunique(dropna=False) # Count unique values, including NaN

    if num_unique == 1:
        constant_columns.append(col)
    elif num_unique == num_rows:
        unique_columns.append(col)

if constant_columns:
    print("Columns with no variability (all values are the same):")
    for col in constant_columns:
        print(f"- {col}: {df[col].iloc[0]}") # Print the constant value
else:
    print("No columns found with no variability (all values are the same).")

print("\n")

if unique_columns:
    print("Columns with high variability (all values are unique):")
    for col in unique_columns:
        print(f"- {col}")
else:
    print("No columns found with high variability (all values are unique).")

In [ ]:
# 5) Export Excel
#df.to_excel("dataset_cleaned.xlsx", index=False)
#from google.colab import files
#files.download("/content/dataset_cleaned.xlsx")

### 1.5 Descriptive Statistics

In [ ]:
# Display descriptive statistics for numerical columns
display(df.describe())

### 1.6 Distribution Analysis of Variables


#### 1.6.1 Continuous Variables

In this first subsection, through some histograms we start by plotting the distributions of the continuous variables.

In [ ]:
# Dataset filtrati
df_viewed_product = df[df['viewed_product'] == 1].copy()
df_added_to_cart = df[df['added_to_cart'] == 1].copy()
df_checkout_started = df[df['started_checkout'] == 1].copy()

# Lista colonne da plottare
continuous_numerical_cols = [
    'ga_session_number',
    'items_viewed',
    'items_added_to_cart',
    'checkout_events',
    'unique_items_viewed',
    'unique_items_added_to_cart',
    'unique_categories_viewed',
    'cart_value',
    'avg_viewed_price',
    'total_engagement_time',
    'session_duration_sec',
    'time_view_to_cart',
    'time_cart_to_checkout',
]


# Tieni solo le colonne esistenti
continuous_numerical_cols = [col for col in continuous_numerical_cols if col in df.columns]

# Costruisco i gruppi
df_full_0 = df[df['purchased'] == 0].copy()
df_full_0['group'] = 'Full - p=0'

df_viewed_0 = df_viewed_product[df_viewed_product['purchased'] == 0].copy()
df_viewed_0['group'] = 'Viewed product - p=0'

df_added_0 = df_added_to_cart[df_added_to_cart['purchased'] == 0].copy()
df_added_0['group'] = 'Cart - p=0'

df_checkout_0 = df_checkout_started[df_checkout_started['purchased'] == 0].copy()
df_checkout_0['group'] = 'Checkout - p=0'

df_purchased_1 = df[df['purchased'] == 1].copy()
df_purchased_1['group'] = 'p=1'

# Unisco tutto
df_plot = pd.concat(
    [df_full_0, df_viewed_0, df_added_0, df_checkout_0, df_purchased_1],
    ignore_index=True
)

group_order = [
    'Full - p=0',
    'Viewed product - p=0',
    'Cart - p=0',
    'Checkout - p=0',
    'p=1'
]

# Palette colori personalizzata
palette = {
    'Full - p=0': 'skyblue',
    'Viewed product - p=0': 'plum',
    'Cart - p=0': 'orange',
    'Checkout - p=0': 'mediumseagreen',
    'p=1': 'salmon'
}

n_plots = len(continuous_numerical_cols)
n_cols = 4
n_rows = math.ceil(n_plots / n_cols)

plt.figure(figsize=(n_cols * 6, n_rows * 5))

for i, col in enumerate(continuous_numerical_cols):
    ax = plt.subplot(n_rows, n_cols, i + 1)

    sns.boxplot(
        data=df_plot,
        x='group',
        y=col,
        hue='group',
        order=group_order,
        hue_order=group_order,
        palette=palette,
        legend=False,
        ax=ax
    )

    ax.set_title(f'Distribution of {col}', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(col, fontsize=10)
    ax.tick_params(axis='x', rotation=25, labelsize=9)
    ax.tick_params(axis='y', labelsize=10)

plt.tight_layout()
plt.show()

The boxplots visually represent the distributions of various continuous variables across different user funnel stages:

*   **`ga_session_number`**: Shows a tendency for higher session numbers in later funnel stages (Checkout and Purchased), indicating that returning users or those with more sessions are more likely to progress. The median generally increases as users move deeper into the funnel. The spread also tends to be wider for earlier stages, suggesting more variability in session history for less committed users, and narrows for converting users.

*   **`items_viewed`** and **`unique_items_viewed`**: Medians generally increase with funnel progression, with 'Checkout - p=0' and 'p=1' showing higher values. This suggests that users who view more items (both total and unique) are more likely to convert. However, the spread for non-purchasers can be wide, indicating some users view many items but don't convert, because they may not be very interested in a real purchase. For 'p=1', the dispersion typically tightens, implying a more consistent pattern of viewing behavior.

*   **`items_added_to_cart`** and **`unique_items_added_to_cart`**: These variables naturally show higher values for stages after 'Viewed product'. The medians are highest for 'p=1' (purchased), suggesting that successful conversions involve adding more items or a greater variety of unique items to the cart. Dispersion in these metrics is generally lower for converting users, indicating more focused cart additions, while abandoners might show a wider range, including extreme outliers for very large carts that are ultimately abandoned.

*   **`checkout_events`**: As expected, this is relevant only for 'Checkout - p=0' and 'p=1', where medians are above zero. The 'p=1' group generally shows a slightly higher median, indicating more checkout interactions for successful purchases. The distributions here are often quite compressed, with few extreme outliers, suggesting that the number of checkout events is generally low.

*   **`unique_categories_viewed`**: Interestingly, this variable tends to show lower medians for 'p=1' (purchased) compared to earlier stages like 'Viewed product - p=0'. This could imply that more focused browsing within fewer categories leads to conversion, while extensive category exploration might indicate indecision. The spread is often broader for non-converting groups, reflecting diverse browsing habits, while converters may show a narrower, more concentrated distribution.

*   **`cart_value`**: This is primarily non-zero for 'Cart - p=0', 'Checkout - p=0', and 'p=1'. The median cart value is generally highest for 'p=1', suggesting that higher-value carts are more likely to convert. 'Checkout - p=0' can also have high cart values, highlighting the issue of high-value cart abandonment, often appearing as outliers in the boxplots. The dispersion can be substantial in all stages where cart value is present, but successful purchases (`p=1`) often show a tighter distribution around a higher median.

*   **`avg_viewed_price`**: Similar to `cart_value`, this tends to be higher for later funnel stages, with 'p=1' generally having the highest median. This indicates that users viewing higher-priced items eventually convert. The distributions can be skewed with upper outliers representing very high-value product views, especially for abandoned carts.

*   **`total_engagement_time`** and **`session_duration_sec`**: These two variables show a clear upward trend in medians as users move deeper into the funnel, with 'p=1' exhibiting the highest values. This strongly suggests that higher engagement and longer session durations are key indicators of intent and ultimately lead to purchase. The distributions are typically right-skewed with significant outliers representing highly engaged or very long sessions, particularly for non-converting users who might be

#### 1.6.2 Categorical Variables


In [ ]:
# Select categorical columns for bar plots
categorical_cols = df.select_dtypes(include=['object', 'string']).columns

# Exclude ID, date, and datetime related columns that are not truly categorical variables for plotting here
exclude_categorical = [
    'user_pseudo_id', 'user_id', 'event_date', 'first_view_datetime', 'first_cart_datetime',
    'first_checkout_datetime', 'purchase_datetime', 'first_view_date', 'first_cart_date',
    'first_checkout_date', 'purchase_date', 'first_view_time', 'first_cart_time',
    'first_checkout_time', 'purchase_time', 'unique_session' # Exclude 'unique_session' as it has too many unique values for a bar plot
]
categorical_cols = [col for col in categorical_cols if col not in exclude_categorical]

# Plot bar plots for categorical columns
plt.figure(figsize=(9, len(categorical_cols) * 4)) # Adjusted figsize for a single column of plots
for i, col in enumerate(categorical_cols):
    plt.subplot(len(categorical_cols), 3, i + 1) # Changed to 1 column for vertical stacking
    sns.countplot(x=df[col].dropna(), order=df[col].value_counts().index, palette='viridis', hue=df[col].dropna(), legend=False) # Changed to x-axis for vertical bars, added hue and legend=False
    plt.title(f'Distribution of {col}')
    plt.xlabel(col) # X-axis will now be the categories
    plt.ylabel('Count') # Y-axis will be the count
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
plt.tight_layout()
plt.show()

The bar plots above visualize the distribution of key categorical variables in the dataset.

*   **`login_status`**: Shows a clear majority of sessions are from 'not logged' users, followed by the 'null' category (which represents sessions where login status was initially missing), and a very small portion from 'loggedin' users. This highlights that a significant portion of user interaction occurs without explicit authentication.

*   **`drop_off_stage`**: This is a crucial variable for our funnel analysis. The plots show the distribution of sessions across different abandonment stages: 'no_product_interaction', 'product_abandonment', 'cart_abandonment', 'checkout_abandonment', and 'converted'. As expected, 'no_product_interaction' and 'product_abandonment' account for the largest shares, indicating where the most users drop off early in the funnel. The 'converted' group is the smallest, representing successful purchases.

*   **`time_of_day_segment`**: This displays the distribution of sessions by time of day (morning, afternoon, evening, night). It reveals peak activity periods, which can be crucial for understanding user behavior and optimizing engagement strategies.

These plots provide a foundational understanding of the user base's characteristics and their progression through the e-commerce funnel based on categorical attributes.

#### 1.6.3 Joint Distributions of Date and Time

In this section, we analyze the joint distribution of event dates and times using 2D histograms. This helps visualize patterns in user activity over time, such as peak hours or specific days for product views, cart additions, checkouts, and purchases. The X-axis represents the date, and the Y-axis represents the time of day. Eventual peaks in the distributions will be used later in the models to account for non observed shock that may be captured by other specific variables.

In [ ]:
pairs = [
    ("first_view_date", "first_view_time", "First View"),
    ("first_cart_date", "first_cart_time", "First Cart"),
    ("first_checkout_date", "first_checkout_time", "First Checkout"),
    ("purchase_date", "purchase_time", "Purchase"),
]

n = len(pairs)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 10))
axes = axes.flatten()

for ax, (date_col, time_col, title) in zip(axes, pairs):
    temp = df[[date_col, time_col]].copy()

    # Convert date
    temp[date_col] = pd.to_datetime(temp[date_col], errors="coerce")

    # Convert time and get minutes of the day
    time_parsed = pd.to_datetime(temp[time_col].astype(str), errors="coerce")
    temp["minutes_in_day"] = time_parsed.dt.hour * 60 + time_parsed.dt.minute

    # Remove missing rows
    temp = temp.dropna(subset=[date_col, "minutes_in_day"]).copy()

    # Convert date to numeric format for plotting
    x = mdates.date2num(temp[date_col])
    y = temp["minutes_in_day"]

    h = ax.hist2d(x, y, bins=[30, 24], cmap="Blues")
    fig.colorbar(h[3], ax=ax, label="Frequency")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.tick_params(axis="x", rotation=45)

    ax.set_yticks([0, 360, 720, 1080, 1380])
    ax.set_yticklabels(["00:00", "06:00", "12:00", "18:00", "23:00"])

    ax.set_xlabel("Date")
    ax.set_ylabel("Time of day")
    ax.set_title(f"Joint distribution: {title}")

# Nasconde eventuali subplot vuoti
for ax in axes[len(pairs):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

The 2D histograms of the joint distribution of dates and times reveal distinct patterns across different stages of the user journey.

In **First View**, the most striking spike appears around **June 11 at about 06:00**, where there is a very dark cell indicating an unusually high concentration of first views. Another strong cluster emerges in **early July, especially between July 6 and July 9, with elevated activity from roughly 12:00 to 18:00**. There is also a visible band around June 30, centered near midday to early afternoon.

In **First Cart**, activity is much lower in absolute terms, but some concentrated moments still stand out. The clearest peak occurs around **July 7, especially between 12:00 and 14:00**, where the darkest cells appear. There is also a smaller but noticeable cluster around June 30 at about 08:00–10:00, and repeated moderate activity in the afternoon/evening of the first week of July.

In **First Checkout**, the pattern becomes more fragmented, but there is still a visible concentration in early July. The **strongest signal appears around July 7 at approximately 12:00**, which seems to be the peak checkout moment in the whole panel. Additional checkout activity is visible around June 29–30, mainly between 08:00 and 17:00, and again on July 8–9 in the late morning / early afternoon.

In **Purchase**, volumes are understandably the lowest, but some specific hotspots are still present. The clearest one is around **July 10 near 12:00**, which appears to be the strongest purchase concentration. There is also a compact cluster between July 6 and July 8, mostly from 09:00 to 17:00, suggesting that the early-July surge in views and carts partially translated into conversions. A few isolated purchases also appear in late June, especially around June 28–30 during midday and afternoon hours.

Taken together, the chart suggests that **early July is the most active period of the whole funnel**, with the strongest concentrations generally occurring between late morning and early evening. The funnel also seems to show a fairly coherent progression: **peaks in First View and First Cart around July 6–8 are followed by visible increases in Checkout and Purchase around July 7–10**.

#### 1.6.4 Comparison between Event Date and first_date variables

To better understand the temporal relationship between the general session event date (`event_date`) and the specific dates when the first interactions occurred (`first_view_date`, `first_cart_date`, `first_checkout_date`, `purchase_date`), we will use scatter plots. These plots will allow us to visualize if these dates are aligned or if there are significant delays between them, providing insight into how quickly users progress through the funnel.

Each plot will show `event_date` on the X-axis and one of the `first_date` variables on the Y-axis. A diagonal `y=x` line will be included to indicate perfect alignment between the two dates. This will help us immediately identify any discrepancies or trends.

In [ ]:
date_pairs = [
    ("event_date", "first_view_date", "Event Date vs First View Date"),
    ("event_date", "first_cart_date", "Event Date vs First Cart Date"),
    ("event_date", "first_checkout_date", "Event Date vs First Checkout Date"),
    ("event_date", "purchase_date", "Event Date vs Purchase Date"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, (event_col, other_col, title) in zip(axes, date_pairs):
    temp = df[[event_col, other_col]].copy()

    temp[event_col] = pd.to_datetime(temp[event_col], errors="coerce")
    temp[other_col] = pd.to_datetime(temp[other_col], errors="coerce")

    temp = temp.dropna(subset=[event_col, other_col]).copy()

    ax.scatter(temp[event_col], temp[other_col], alpha=0.25)

    min_date = min(temp[event_col].min(), temp[other_col].min())
    max_date = max(temp[event_col].max(), temp[other_col].max())
    ax.plot([min_date, max_date], [min_date, max_date], linestyle="--")

    ax.set_xlabel(event_col)
    ax.set_ylabel(other_col)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=45)

plt.tight_layout()
plt.show()

These charts show a very clear pattern: all funnel dates are **almost perfectly aligned with the event date**.

More specifically:

*   **First View Date vs Event Date:** The points lie almost exactly on the diagonal, which suggests that the **first view happens on the same day as the event** in nearly all cases.

*   **First Cart Date vs Event Date:** The same pattern appears here, with **very little dispersion**. This means users are typically **adding items to the cart immediately or with only negligible delay**.

*   **First Checkout Date vs Event Date:** This plot shows slightly more variability, but the spread is still very limited. So **checkout also tends to happen on the same day or with only a minimal lag**.

*   **Purchase Date vs Event Date:** This is the cleanest chart of all. The points are **almost perfectly superimposed on the diagonal**, indicating that **purchases are also strongly synchronized with the event date**.

Overall, the charts suggest that the funnel is **highly compressed in time**:

*   There are **no meaningful delays** between view, cart, checkout, and purchase;

*   There are **no major outliers or long-tail behaviors**;

*   The event appears to be **very closely tied to conversion activity**.

**One useful note:** Such a near-perfect alignment is interesting and may reflect genuinely immediate user behavior, but it is also worth validating from a data perspective. It could mean:

*   Users really **convert very quickly**,

*   The data is **aggregated at the daily level**,

*   Or **multiple funnel milestones inherit the same recorded date** due to tracking logic.

#### Engagement time by session

In [ ]:
df["total_engagement_time"] = (
    df["total_engagement_time"]
    .replace(["", " ", "null", "NULL", "None", "none", "NA", "N/A", "nan"], pd.NA)
)

print("True nulls after cleaning:", df["total_engagement_time"].isna().sum())

# Create Null / Non Null variable
temp = df[["drop_off_stage", "total_engagement_time"]].copy()
temp["engagement_status"] = temp["total_engagement_time"].isna().map({
    True: "Null",
    False: "Non Null"
})

# Frequency table
count_data = pd.crosstab(temp["drop_off_stage"], temp["engagement_status"])

# Percentages of total per category (row)
plot_data_pct = count_data.div(count_data.sum(axis=1), axis=0) * 100

print(plot_data_pct)

# Plot
plot_data_pct.plot(kind="bar", figsize=(10, 6))
plt.title("Percentage of total_engagement_time null and non-null by drop_off_stage")
plt.xlabel("drop_off_stage")
plt.ylabel("Percentage")
plt.xticks(rotation=45)
plt.legend(title="total_engagement_time")
plt.tight_layout()
plt.show()

## 2 Feature engineering

We now want to expand the size of the dataset by exploring the possibility of adding new variables to the dataset thourgh feature engineering. In some subsections, the creation of new variables in supported by some plots, to better explain if, beyond our reasoning, the addition of new variables is useful to the purposes of the analysis.

The insights for creating the varables are not suggested by compicated code but rather by a simpler visual analysis on excel.

### 2.1 Logarithmic Transformation of existing variables

In this first subsection, we create the logarithmic transformation of those variables that exhibited high skewness in section 1.5.1.

In [ ]:
# List of variables identified with high skewness from section 1.5.1
high_skewness_vars = [
    'ga_session_number',
    'items_viewed',
    'items_added_to_cart',
    'checkout_events',
    'unique_items_viewed',
    'unique_items_added_to_cart',
    'unique_items_purchased',
    'unique_categories_viewed',
    'cart_value',
    'avg_viewed_price',
    'total_engagement_time', # Included as requested
    'session_duration_sec',  # Included as requested
    'time_view_to_cart',
    'time_cart_to_checkout'
    # 'time_checkout_to_purchase' removed as it was not in the 1.5.1 plotting list
]

# Apply np.log1p transformation and create new columns
for var in high_skewness_vars:
    # Ensure the column is numeric before transformation, coercing errors to NaN
    df[var] = pd.to_numeric(df[var], errors='coerce')
    # Only apply transformation if the column exists and has non-null values
    if var in df.columns and df[var].isnull().sum() < len(df):
        # For time variables, ensure non-negative values for log1p
        if 'time_' in var:
            df[var] = df[var].apply(lambda x: x if x >= 0 else np.nan)
        df[f'log_{var}'] = np.log1p(df[var])
        print(f"Created log_transformed column for '{var}'")

# Drop the original high skewness variables
# We will only drop if the log-transformed version was successfully created
columns_to_drop = [var for var in high_skewness_vars if f'log_{var}' in df.columns]

# Exclude specific variables from being dropped as they are needed for ratio calculations
columns_to_retain_original = [
    'items_viewed',
    'unique_items_viewed',
    'items_added_to_cart',
    'unique_items_added_to_cart',
    'ga_session_number',
    'total_engagement_time',
    'session_duration_sec'
]

# Remove retained columns from the list of columns to drop
columns_to_drop = [col for col in columns_to_drop if col not in columns_to_retain_original]

if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
else:
    print("No additional original highly skewed variables were dropped.")

print("\nOriginal highly skewed variables dropped (excluding specified retained ones).")
print(f"DataFrame now has {df.shape[1]} columns.")

display(df.head())

In [ ]:
# Dataset filtrati
df_viewed_product = df[df['viewed_product'] == 1].copy()
df_added_to_cart = df[df['added_to_cart'] == 1].copy()
df_checkout_started = df[df['started_checkout'] == 1].copy()

# Lista colonne da plottare
log_transformed_cols = [
    'log_ga_session_number',
    'log_items_viewed',
    'log_items_added_to_cart',
    'log_checkout_events',
    'log_unique_items_viewed',
    'log_unique_items_added_to_cart',
    'log_unique_categories_viewed',
    'log_cart_value',
    'log_avg_viewed_price',
    'log_total_engagement_time',
    'log_session_duration_sec',
    'log_time_view_to_cart',
    'log_time_cart_to_checkout',
]

# Tieni solo le colonne esistenti
continuous_numerical_cols = [col for col in log_transformed_cols if col in df.columns]

# Costruisco i gruppi
df_full_0 = df[df['purchased'] == 0].copy()
df_full_0['group'] = 'Full - p=0'

df_viewed_0 = df_viewed_product[df_viewed_product['purchased'] == 0].copy()
df_viewed_0['group'] = 'Viewed product - p=0'

df_added_0 = df_added_to_cart[df_added_to_cart['purchased'] == 0].copy()
df_added_0['group'] = 'Cart - p=0'

df_checkout_0 = df_checkout_started[df_checkout_started['purchased'] == 0].copy()
df_checkout_0['group'] = 'Checkout - p=0'

df_purchased_1 = df[df['purchased'] == 1].copy()
df_purchased_1['group'] = 'p=1'

# Unisco tutto
df_plot = pd.concat(
    [df_full_0, df_viewed_0, df_added_0, df_checkout_0, df_purchased_1],
    ignore_index=True
)

group_order = [
    'Full - p=0',
    'Viewed product - p=0',
    'Cart - p=0',
    'Checkout - p=0',
    'p=1'
]

# Palette colori personalizzata
palette = {
    'Full - p=0': 'skyblue',
    'Viewed product - p=0': 'plum',
    'Cart - p=0': 'orange',
    'Checkout - p=0': 'mediumseagreen',
    'p=1': 'salmon'
}

n_plots = len(continuous_numerical_cols)
n_cols = 4
n_rows = math.ceil(n_plots / n_cols)

plt.figure(figsize=(n_cols * 8, n_rows * 5))

for i, col in enumerate(continuous_numerical_cols):
    ax = plt.subplot(n_rows, n_cols, i + 1)

    sns.boxplot(
        data=df_plot,
        x='group',
        y=col,
        hue='group',
        order=group_order,
        hue_order=group_order,
        palette=palette,
        legend=False,
        ax=ax
    )

    ax.set_title(f'Distribution of {col}', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(col, fontsize=10)
    ax.tick_params(axis='x', rotation=25, labelsize=9)
    ax.tick_params(axis='y', labelsize=10)

plt.tight_layout()
plt.show()

**Analysis of Log-Transformed Continuous Variables Across Funnel Stages**

The boxplots displayed above allow for a direct comparison of variable distributions across five key user segments:

*   **`Full - p=0` (skyblue):** Sessions that did not result in a purchase, from the entire dataset.
*   **`Viewed product - p=0` (plum):** Sessions that viewed a product but did not purchase.
*   **`Cart - p=0` (orange):** Sessions that reached the 'added to cart' stage but did not purchase.
*   **`Checkout - p=0` (mediumseagreen):** Sessions that reached the 'started checkout' stage but did not purchase.
*   **`p=1` (salmon):** All sessions that resulted in a purchase.

This comparative view helps identify how key metrics shift as users progress (or abandon) through the funnel. Observing these shifts is crucial for understanding the characteristics of engaged, converting users versus those who drop off.

Here are the significant changes in distributions, focusing on medians, spread, and outliers:

*   **`log_total_engagement_time` and `log_session_duration_sec`:** These variables show a **strong and consistent upward trend in median values** across the funnel. The `p=1` group (purchasers) consistently exhibits the highest medians, followed by `Checkout - p=0`, `Cart - p=0`, and then `Viewed product - p=0`, with `Full - p=0` showing the lowest. This indicates that deeper engagement and longer session duration are increasingly critical indicators of progression. The interquartile range (IQR) for `p=1` is generally narrower, indicating more focused engagement for successful conversions, while the `Full - p=0` and `Viewed product - p=0` groups often have a wider spread, reflecting a broader range of less committed behaviors. Outliers are present in all groups but are generally shifted to higher values for later funnel stages.

*   **`log_cart_value` and `log_avg_viewed_price`:** As expected, `log_cart_value` shows a significant increase in median from `Cart - p=0` to `Checkout - p=0` and especially to `p=1`. This indicates that successful purchases involve higher cart values. `log_avg_viewed_price` also generally increases, though less dramatically, with `p=1` typically having slightly higher medians. The spread for these variables tends to be narrower for `p=1` and `Checkout - p=0`, suggesting more concentrated price points or cart values among high-intent users. Interestingly, `Checkout - p=0` (abandoners) can show considerable outliers at higher cart values, indicating instances of high-value cart abandonment.

*   **`log_items_viewed`, `log_items_added_to_cart`, `log_unique_items_viewed`, `log_unique_items_added_to_cart`:** These interaction metrics generally show increasing medians as users move deeper into the funnel, with `p=1` usually having the highest. This confirms that more interaction (views, adds to cart) correlates with progression. Notably, `Viewed product - p=0` already shows higher medians than `Full - p=0` for viewing-related metrics, as expected. However, the dispersion (IQR) can be quite wide, particularly for `Cart - p=0` and `Checkout - p=0` groups, implying diverse interaction patterns even among those who abandon. For example, some non-purchasers might have very high interaction counts, suggesting indecision.

*   **`log_unique_categories_viewed`:** The median of this variable tends to be higher for earlier stages (`Full - p=0` and `Viewed product - p=0`) and often slightly *lower* for `p=1`. This suggests that extensive browsing across many unique categories (higher values) might indicate less focused intent, potentially leading to abandonment, while converters (`p=1`) might have more targeted browsing behavior. The spread is generally high across all non-purchasing groups.

*   **`log_checkout_events`:** This variable exhibits a clear and expected progression. Its median is very low (often near zero) for `Full - p=0`, `Viewed product - p=0`, and `Cart - p=0`. It then significantly increases for `Checkout - p=0` and reaches its highest median for `p=1`. The spread for `p=1` is often tighter, indicating a more consistent number of checkout interactions for successful conversions.

*   **`log_time_view_to_cart` and `log_time_cart_to_checkout`:** For these 'time-to-event' variables, `p=1` (purchasers) generally shows **lower median values**, indicating quicker decision-making and less hesitation between stages. For example, `log_time_view_to_cart` often has a lower median for `p=1` compared to `Cart - p=0`, implying that quicker decisions to add to cart are associated with eventual purchase. The dispersion might be narrower for `p=1`, suggesting a more consistent and efficient funnel progression for these users.

In summary, the combined boxplots visually reinforce the characteristics of high-intent users: generally higher engagement times, higher cart values, more focused browsing (fewer unique categories viewed extensively), and quicker progression through critical funnel stages. While the medians clearly show these trends, the varying dispersion and presence of outliers provide valuable nuances, indicating that user behavior is complex and not all abandoners fit a single profile.

### 2.2 User ID Categorization

After we exported the file on excel, we saw that the variable `user_id` had several entries that could be grouped in 4 categories: null values, numerical values, letter E + numerical values and EMP + numerical values.

Since having multiple alphanumerical entries is not very useful to carry out a proper analysis, we look for the possibility of creating a categorical variable following the above strategy.

In [ ]:
# Function to classify user_id
def classify_user_id(user_id):
    # Check for NaN or '<NA>' string representation
    if pd.isna(user_id) or str(user_id) == '<NA>':
        return 'NULL'

    user_id_str = str(user_id)
    if user_id_str.startswith('EMP'):
        return 'EMP'
    elif user_id_str.startswith('E'):
        return 'E'
    elif user_id_str.isnumeric():
        return 'NUM'
    else:
        return 'OTHER' # Catch-all for other patterns not explicitly defined

# Apply the function to create the new categorical column
df['user_id_category'] = df['user_id'].apply(classify_user_id)

# Display the value counts of the new column
print("Value counts for 'user_id_category':")
display(df['user_id_category'].value_counts())

The code shows us very unbalanced results. In particular, null values form more than 98% of the observations, while the other 3 categories appear very rarely (numerical entries appear approximately 1% of the times, while the other 2 basically never come up).

For this reason, it is better to reduce the unbalancedness of the entries frequencies (at least partially) and combine the 3 least frequent categories in a new one.

In [ ]:
# Create the user_id_binary column
df['user_id_binary'] = df['user_id_category'].apply(lambda x: 1 if x in ['NUM', 'E', 'EMP'] else 0)

# Display the value counts of the new column
print("Value counts for 'user_id_binary':")
display(df['user_id_binary'].value_counts())

Our final variable is `user_id_binary`, which takes value 1 when the `user_id` entry is not null and 0 otherwise. The percentage of 1's in the columns is 1.1%, while di rpoportion of 0's is approximately 98.6%.

### 2.3 User and Session ID Analysis

In [ ]:
# Analyze user_pseudo_id
total_user_pseudo_id = len(df['user_pseudo_id'])
unique_user_pseudo_id = df['user_pseudo_id'].nunique()

print(f"Total entries for 'user_pseudo_id': {total_user_pseudo_id}")
print(f"Unique entries for 'user_pseudo_id': {unique_user_pseudo_id}\n")

# Analyze session_id
total_session_id = len(df['session_id'])
unique_session_id = df['session_id'].nunique()

print(f"Total entries for 'session_id': {total_session_id}")
print(f"Unique entries for 'session_id': {unique_session_id}")

In [ ]:
# Calculate the frequency of each user_pseudo_id
user_pseudo_id_counts = df['user_pseudo_id'].value_counts()

# Create the user_pseudo_id_unique binary column
df['user_pseudo_id_unique'] = df['user_pseudo_id'].map(user_pseudo_id_counts).apply(lambda x: 1 if x == 1 else 0)

print("Value counts for 'user_pseudo_id_unique':")
display(df['user_pseudo_id_unique'].value_counts())

print("\n")

# Calculate the frequency of each session_id
session_id_counts = df['session_id'].value_counts()

# Create the session_id_unique binary column
df['session_id_unique'] = df['session_id'].map(session_id_counts).apply(lambda x: 1 if x == 1 else 0)

print("Value counts for 'session_id_unique':")
display(df['session_id_unique'].value_counts())

We created the unique_session column by concatenating user_pseudo_id and session_id. This helps accurately identify distinct user sessions, especially when session_id might be reused or inconsistent across time, ensuring each visit is uniquely tracked for analysis.

In [ ]:
# To identify all the unique sessions (since the variable "session_id" relies on the time stamp),
# we create a variable "unique_session" that concatenates user_pseudo_id and session_id

df['unique_session'] = df['user_pseudo_id'].astype(str)+ '_' + df['session_id'].astype(str)
df[['user_pseudo_id', 'session_id', 'unique_session']].head()

In [ ]:
num_unique_sessions = df['unique_session'].nunique()
total_rows = len(df)

print(f"Number of unique sessions: {num_unique_sessions}")
print(f"Total number of records in the DataFrame: {total_rows}")
print(f"Average number of events per session: {total_rows/ num_unique_sessions:.4f}")

# One session generates multiple events, we have an average of 57 events per session

### 2.4 Ratios of Items and Unique Items

In this subsection, we explore the creation of two new variables: `ratio_items_viewed_unique` and `ratio_items_added_to_cart_unique`. These ratios are derived by dividing `items_viewed` by `unique_items_viewed`, and `items_added_to_cart` by `unique_items_added_to_cart`, respectively.

These variables can serve as a measure of user decisiveness. Users with a ratio value close to 1 might be considered more focused in their browsing and purchasing decisions. Conversely, higher ratio values could suggest less focused behavior, where users repeatedly interact with the same items without immediately progressing through the funnel.

In [ ]:
# Calculate the ratio of items viewed to unique items viewed
df['ratio_items_viewed_unique'] = df.apply(lambda row: row['items_viewed'] / row['unique_items_viewed'] if row['unique_items_viewed'] > 0 else 0, axis=1)

# Calculate the ratio of items added to cart to unique items added to cart
df['ratio_items_added_to_cart_unique'] = df.apply(lambda row: row['items_added_to_cart'] / row['unique_items_added_to_cart'] if row['unique_items_added_to_cart'] > 0 else 0, axis=1)

print("New ratio variables 'ratio_items_viewed_unique' and 'ratio_items_added_to_cart_unique' created.")
display(df[['items_viewed', 'unique_items_viewed', 'ratio_items_viewed_unique', 'items_added_to_cart', 'unique_items_added_to_cart', 'ratio_items_added_to_cart_unique']].head())

These new variables, `ratio_items_viewed_unique` and `ratio_items_added_to_cart_unique`, represent the average number of times each unique item was viewed or added to the cart, respectively. A higher ratio indicates that users interacted with the same items multiple times. These can be important indicators of user engagement and intent.

In [ ]:
vars_to_plot_ratios = [
    'ratio_items_viewed_unique',
    'ratio_items_added_to_cart_unique'
]

n_vars_ratios = len(vars_to_plot_ratios)
n_cols_ratios = 2
n_rows_ratios = math.ceil(n_vars_ratios / n_cols_ratios)

fig_ratios, axes_ratios = plt.subplots(n_rows_ratios, n_cols_ratios, figsize=(12, 4 * n_rows_ratios))
fig_ratios.suptitle('Distributions of Ratio Variables', fontsize=14)

# Ensure axes_ratios is always a flat array for iteration
if n_vars_ratios == 1:
    axes_ratios = np.array([axes_ratios]) # Wrap single Axes object in an array
else:
    axes_ratios = axes_ratios.flatten() # Flatten 1D or 2D array of Axes

for i, var in enumerate(vars_to_plot_ratios):
    ax = axes_ratios[i]
    ax.hist(df[var].dropna(), bins=30, color='#4C78A8', alpha=0.8, edgecolor='white') # Changed color to blue
    ax.set_title(f'{var}\nskewness: {df[var].skew():.2f}', fontsize=10)
    ax.set_ylabel('Frequency')
    ax.set_xlabel(var)

# Hide any unused subplots
for j in range(len(vars_to_plot_ratios), len(axes_ratios)):
    axes_ratios[j].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Now re-plot the distributions of the ratio variables after filtering out sessions with `no_product_interaction` and `product_abandonment` to focus on more advanced interaction stages.

As it happened with the other continuous variables in the dataset, the distribution is still very skewed, which suggest that we could take a logarithmic transformation to better deal with these variables.

In [ ]:
vars_to_plot_ratios = [
    'ratio_items_viewed_unique',
    'ratio_items_added_to_cart_unique'
]

# Temporarily filter for plotting, but do not create a new permanent dataframe
df_temp_plot = df[(df['drop_off_stage'] != 'no_product_interaction') & (df['drop_off_stage'] != 'product_abandonment')].copy()

n_vars_ratios = len(vars_to_plot_ratios)
n_cols_ratios = 2
n_rows_ratios = math.ceil(n_vars_ratios / n_cols_ratios)

fig_ratios, axes_ratios = plt.subplots(n_rows_ratios, n_cols_ratios, figsize=(12, 4 * n_rows_ratios))
fig_ratios.suptitle('Distributions of Ratio Variables (Filtered for Plotting)', fontsize=14)

# Ensure axes_ratios is always a flat array for iteration
if n_vars_ratios == 1:
    axes_ratios = np.array([axes_ratios]) # Wrap single Axes object in an array
else:
    axes_ratios = axes_ratios.flatten() # Flatten 1D or 2D array of Axes

for i, var in enumerate(vars_to_plot_ratios):
    ax = axes_ratios[i]
    ax.hist(df_temp_plot[var].dropna(), bins=30, color='#4C78A8', alpha=0.8, edgecolor='white') # Use df_temp_plot
    ax.set_title(f'{var}\nskewness: {df_temp_plot[var].skew():.2f}', fontsize=10) # Use df_temp_plot
    ax.set_ylabel('Frequency')
    ax.set_xlabel(var)

# Hide any unused subplots
for j in range(len(vars_to_plot_ratios), len(axes_ratios)):
    axes_ratios[j].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Both ratio variables exhibit **extreme right skewness** — skewness of 17.19 for
`ratio_items_viewed_unique` and 6.87 for `ratio_items_added_to_cart_unique` — with
the vast majority of sessions concentrated at ratio = 1 (meaning each unique item
was interacted with exactly once). The long right tail reflects a small minority of
sessions where users repeatedly viewed or added the same items multiple times.

This pattern is consistent with typical e-commerce browsing behavior: most users
scan products once and move on, while a small subset revisits the same items
repeatedly — a behavioral signal that could indicate either high interest or
indecision. The severity of the skewness (particularly 17.19) makes these variables
unsuitable for direct use in linear models, motivating the log-transformation and
binary dummy creation that follow.

#### Logarithmic Transformation of Ratio Variables

As observed from the previous histograms, both `ratio_items_viewed_unique` and `ratio_items_added_to_cart_unique` exhibit strong positive skewness, with a heavy concentration of values near zero. Such skewed distributions can negatively impact the performance of certain statistical models or algorithms that assume normally distributed data.

To address this, we will apply a logarithmic transformation using `np.log1p`. The `log1p` function computes `log(1+x)`, which is particularly useful for data that includes zero values, as `log(0)` is undefined. This transformation helps in reducing skewness, making the distributions more symmetric and closer to a normal distribution, which can improve the efficacy of subsequent analyses.

In [ ]:
import numpy as np

# Apply log1p transformation to the ratio variables in the DataFrame
df['log_ratio_items_viewed_unique'] = np.log1p(df['ratio_items_viewed_unique'])
df['log_ratio_items_added_to_cart_unique'] = np.log1p(df['ratio_items_added_to_cart_unique'])

print("Log-transformed ratio variables created on the original DataFrame.")
display(df[['ratio_items_viewed_unique', 'log_ratio_items_viewed_unique', 'ratio_items_added_to_cart_unique', 'log_ratio_items_added_to_cart_unique']].head())

In [ ]:
vars_to_plot_log_ratios = [
    'log_ratio_items_viewed_unique',
    'log_ratio_items_added_to_cart_unique'
]

# Temporarily filter for plotting, but do not create a new permanent dataframe
df_temp_plot = df[(df['drop_off_stage'] != 'no_product_interaction') & (df['drop_off_stage'] != 'product_abandonment')].copy() # Filter for plotting only

n_vars_log_ratios = len(vars_to_plot_log_ratios)
n_cols_log_ratios = 2
n_rows_log_ratios = math.ceil(n_vars_log_ratios / n_cols_log_ratios)

fig_log_ratios, axes_log_ratios = plt.subplots(n_rows_log_ratios, n_cols_log_ratios, figsize=(12, 4 * n_rows_log_ratios))
fig_log_ratios.suptitle('Distributions of Log-Transformed Ratio Variables (Filtered for Plotting)', fontsize=14)

# Ensure axes_log_ratios is always a flat array for iteration
if n_vars_log_ratios == 1:
    axes_log_ratios = np.array([axes_log_ratios]) # Wrap single Axes object in an array
else:
    axes_log_ratios = axes_log_ratios.flatten() # Flatten 1D or 2D array of Axes

for i, var in enumerate(vars_to_plot_log_ratios):
    ax = axes_log_ratios[i]
    ax.hist(df_temp_plot[var].dropna(), bins=30, color='#6C3483', alpha=0.8, edgecolor='white') # Use df_temp_plot
    ax.set_title(f'{var}\nskewness: {df_temp_plot[var].skew():.2f}', fontsize=10) # Use df_temp_plot
    ax.set_ylabel('Frequency')
    ax.set_xlabel(var)

# Hide any unused subplots
for j in range(len(vars_to_plot_log_ratios), len(axes_log_ratios)):
    axes_log_ratios[j].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

The log-transformation substantially reduces skewness for both variables —
from 17.19 to **2.53** for `log_ratio_items_viewed_unique` and from 6.87 to
**3.96** for `log_ratio_items_added_to_cart_unique`. However, significant
skewness persists in both cases, particularly for `log_ratio_items_added_to_cart_unique`
which retains a sharp spike at ~0.75 (corresponding to ratio = 1 in the original
scale, i.e. `log1p(1) ≈ 0.693`).

This residual skewness indicates that the log-transformation alone is insufficient
to produce well-behaved continuous distributions suitable for linear models. The
mass of sessions at a single point (ratio = 1) creates a semi-discrete distribution
that no monotonic transformation can fully normalize. This motivates the creation
of binary dummy variables in the next step, which directly captures the meaningful
behavioral distinction between users who interacted with each unique item exactly
once (ratio = 1) versus those who revisited the same items multiple times (ratio > 1).

#### Creation of Dummy Variables for High Skewness Ratios

Even after applying a logarithmic transformation, the distributions of `log_ratio_items_viewed_unique` and `log_ratio_items_added_to_cart_unique` still exhibit significant skewness. This indicates that a large proportion of sessions continue to have low ratio values (meaning users interact with unique items only once or very few times), while a smaller number of sessions show much higher ratios.

To address this persistent skewness and capture the distinction between users who repeatedly engage with the same items versus those who do not, we will create two new binary (dummy) variables:

*   `"ratio_items_viewed>1"`: This variable will be `1` if `ratio_items_viewed_unique` is greater than 1 (indicating repeated views of the same unique item), and `0` otherwise.
*   `"ratio_items_added_to_cart>1"`: This variable will be `1` if `ratio_items_added_to_cart_unique` is greater than 1 (indicating repeated additions to cart of the same unique item), and `0` otherwise.

These dummy variables will simplify the interpretation of user behavior, allowing us to directly analyze the impact of repeated item interactions, rather than dealing with the highly skewed continuous distributions.

In [ ]:
# Create dummy variables for ratios greater than 1
df["ratio_items_viewed>1"] = (df['ratio_items_viewed_unique'] > 1).astype(int)
df["ratio_items_added_to_cart>1"] = (df['ratio_items_added_to_cart_unique'] > 1).astype(int)

print("Dummy variables created on the original DataFrame:")
display(df[['ratio_items_viewed_unique', "ratio_items_viewed>1",
                         'ratio_items_added_to_cart_unique', "ratio_items_added_to_cart>1"]].head())

In [ ]:
dummy_vars_to_plot = [
    "ratio_items_viewed>1",
    "ratio_items_added_to_cart>1"
]

# Temporarily filter for plotting, but do not create a new permanent dataframe
df_temp_plot = df[(df['drop_off_stage'] != 'no_product_interaction') & (df['drop_off_stage'] != 'product_abandonment')].copy() # Filter for plotting only

n_dummy_vars = len(dummy_vars_to_plot)
n_cols_dummy = 2
n_rows_dummy = math.ceil(n_dummy_vars / n_cols_dummy)

fig_dummy, axes_dummy = plt.subplots(n_rows_dummy, n_cols_dummy, figsize=(12, 4 * n_rows_dummy))
fig_dummy.suptitle('Distributions of Dummy Ratio Variables (Filtered for Plotting)', fontsize=14) # Update title

# Ensure axes_dummy is always a flat array for iteration
if n_dummy_vars == 1:
    axes_dummy = np.array([axes_dummy])
else:
    axes_dummy = axes_dummy.flatten()

for i, var in enumerate(dummy_vars_to_plot):
    ax = axes_dummy[i]
    sns.countplot(x=df_temp_plot[var], ax=ax, palette='mako', hue=df_temp_plot[var], legend=False) # Use df_temp_plot
    ax.set_title(f'Count of {var}', fontsize=10)
    ax.set_ylabel('Count')
    ax.set_xlabel(var)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Ratio = 1', 'Ratio > 1']) # This was the user's previous edit, ensure it stays

# Hide any unused subplots
for j in range(len(dummy_vars_to_plot), len(axes_dummy)):
    axes_dummy[j].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

dummy_vars_to_plot = [
    "ratio_items_viewed>1",
    "ratio_items_added_to_cart>1"
]

#### Concluding Remarks on Dummy Variable Creation

By transforming the highly skewed continuous ratio variables (`ratio_items_viewed_unique` and `ratio_items_added_to_cart_unique`) into binary dummy variables, we have achieved a more interpretable representation of user behavior. The new variables, `ratio_items_viewed_unique_greater_than_1` and `ratio_items_added_to_cart_unique_greater_than_1`, categorize sessions into two clear groups: those where unique items were interacted with only once (ratio = 1) and those where unique items were interacted with multiple times (ratio > 1).

As seen in the count plots, these dummy variables now provide a discrete distribution, making it easier to analyze the proportion of users exhibiting each behavior. This approach simplifies the analysis of the impact of repeated item interactions, moving away from the complexities of highly skewed continuous distributions and offering a clearer, actionable insight into user engagement patterns.

### 2.5 Cleaning of login_status

In this subsection, we modify the structure of the variable `login_status` to transform the null values into a third category called "null". This modification allows us to avoid throwing away observations that have null values in that column.

In [ ]:
df['login_status'] = df['login_status'].fillna('null')

print("Null values in 'login_status' replaced with 'null'.")
print(df['login_status'].value_counts())

In [ ]:
# Plotting the counts of 'login_status' categories
plt.figure(figsize=(8, 5))
sns.countplot(x=df['login_status'], order=df['login_status'].value_counts().index, palette='viridis', hue=df['login_status'], legend=False)
plt.title('Distribution of Login Status Categories (including Null)')
plt.xlabel('Login Status')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

The `login_status` variable has now been cleaned, with all previously null values explicitly labeled as 'null'. The count plot clearly shows the distribution across the three categories: 'not logged', 'logged', and 'null'. As observed, 'not logged' sessions are the most frequent, followed by the newly defined 'null' category, and finally, 'logged' sessions are the least frequent. This transformation makes the column more robust for analysis by explicitly handling missing values as a distinct category, rather than treating them as unknown or dropping them.

In [ ]:
# Create a new dummy variable: 1 for 'loggedin' or 'null', 0 for 'not logged'
df['logged_in_or_null_dummy'] = df['login_status'].apply(lambda x: 1 if x in ['loggedin', 'null'] else 0)

print("Value counts for 'logged_in_or_null_dummy':")
display(df['logged_in_or_null_dummy'].value_counts())

In [ ]:
# Plotting the counts of the new dummy variable
plt.figure(figsize=(7, 5))
sns.countplot(x=df['logged_in_or_null_dummy'], palette='pastel', hue=df['logged_in_or_null_dummy'], legend=False)
plt.title('Distribution of Logged In or Null Status')
plt.xlabel('Logged In or Null (1) vs Not Logged (0)')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['Not Logged (0)', 'Logged In or Null (1)'])
plt.tight_layout()
plt.show()

The new dummy variable, `logged_in_or_null_dummy`, has been created to categorize users into two main groups: those who are 'not logged in' (0) and those who are 'logged in' or had a 'null' status (1). This grouping is useful for simplifying the analysis, especially considering that the 'loggedin' category was much less frequent than the 'null' category.

From the bar chart, we can clearly see the distribution of these two groups. Most sessions fall into the 'not logged in' category (0), while a smaller, but significant, portion falls into the combined 'logged in or null' category (1). This confirms that a large part of website interactions occur with unidentified or unauthenticated users, providing a clearer view of general user behavior in relation to login status.

### 2.6 Pseudo_user_id and Session_id

In this subsection we want to focus on the analysis of `pseudo_user_id` and `session_id`. From the description in the data dictionary, there may be more sessions for each user, but each user has at least one session. Since working with strings is not useful to extract valuable information, we look for the possibility of creating a dummy or categorical variable that summarizes the relationship between these 2 variables.

In [ ]:
# Group by user_pseudo_id and count unique session_ids
sessions_per_user = df.groupby('user_pseudo_id')['session_id'].nunique()
df['num_sessions_per_user'] = df['user_pseudo_id'].map(sessions_per_user)

# Count the frequency of each session count
session_count_distribution = sessions_per_user.value_counts().sort_index()

print("Distribution of sessions per user_pseudo_id:")
display(session_count_distribution)

In [ ]:
# Plot distribution of sessions per user (capped at 10 for readability)
plot_data = session_count_distribution[session_count_distribution.index <= 10]

plt.figure(figsize=(10, 5))
plt.bar(plot_data.index, plot_data.values, color='steelblue', edgecolor='white')
plt.xlabel('Number of sessions per user')
plt.ylabel('Number of users')
plt.title('Distribution of Sessions per User (capped at 10 for readability)')
plt.xticks(range(1, 11))
plt.tight_layout()
plt.show()

print(f"Users with exactly 1 session: {session_count_distribution[1]:,} ({session_count_distribution[1]/session_count_distribution.sum()*100:.1f}%)")
print(f"Users with 2+ sessions: {session_count_distribution[session_count_distribution.index > 1].sum():,} ({session_count_distribution[session_count_distribution.index > 1].sum()/session_count_distribution.sum()*100:.1f}%)")
print(f"Max sessions for a single user: {session_count_distribution.index.max()}")

The distribution of sessions per user is **heavily right-skewed**: the majority
of users (98,355 out of ~122,000 unique users, approximately **80%**) appear in
the dataset with only a single session, while a long tail of returning users
contributes multiple sessions — with the most active user recorded across 240
distinct sessions.

This panel structure has two important implications for the analysis:

1. **User-level train/test split is essential** (implemented in Section 3.2):
   a random row-level split would allow the same user to appear in both train
   and test sets, causing data leakage and artificially inflating model performance.

2. **The user history features created in Section 2.6** (`had_*_before`,
   `n_*_before`) are meaningful precisely because a non-trivial share of users
   (~20%) return across multiple sessions, making their prior behavioral history
   a legitimate and informative predictor of current-session conversion.

The variable `num_sessions_per_user` is retained as a direct feature in the
models to capture this longitudinal engagement dimension.

### 2.7 Historical Session Data

In this section, we prepare historical session data by creating new features that capture a user's past interactions. This involves identifying a unique user key, parsing event dates, and generating flags to indicate whether a user had performed specific actions (like viewing a product, adding to cart, starting checkout, or purchasing) in previous sessions. We also calculate the number of times a user had performed these actions before the current session.

These historical features are crucial for models that aim to predict future user behavior, as past actions often provide strong indicators of future intent and engagement.

In [ ]:
# Behavioural columns to track historically
behaviour_cols = ["viewed_product", "added_to_cart", "started_checkout", "purchased"]

# User key: use user_id if available, otherwise fall back to user_pseudo_id
df["user_key"] = df["user_id"].fillna(df["user_pseudo_id"]).astype(str)

# Parse event date
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

# Ensure behavioural flags are binary integers
for col in behaviour_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Sort sessions chronologically within each user
df = df.sort_values(
    ["user_key", "event_date", "ga_session_number", "session_id"]
).reset_index(drop=True)

# Binary flag: did the user perform this action in any prior session?
for col in behaviour_cols:
    shifted = df.groupby("user_key")[col].shift(1).fillna(0)
    df[f"had_{col}_before"] = (
        shifted.groupby(df["user_key"]).cummax().astype(int)
    )

# Count: how many times had the user performed this action before this session?
for col in behaviour_cols:
    shifted = df.groupby("user_key")[col].shift(1).fillna(0)
    df[f"n_{col}_before"] = (
        shifted.groupby(df["user_key"]).cumsum().astype(int)
    )

# Preview
cols_to_show = [
    "user_key", "session_id", "event_date",
    "viewed_product", "added_to_cart", "started_checkout", "purchased",
    "had_viewed_product_before", "had_added_to_cart_before",
    "had_started_checkout_before", "had_purchased_before",
    "n_viewed_product_before", "n_added_to_cart_before",
    "n_started_checkout_before", "n_purchased_before"
]
# Show one example user with multiple sessions to verify the logic
example_user = df[df["num_sessions_per_user"] > 3]["user_key"].iloc[0]
example_df = df[df["user_key"] == example_user][cols_to_show]
print(f"Example user: {example_user} ({len(example_df)} sessions)")
display(example_df)

# Summary statistics on history features
print("\nShare of sessions with prior activity:")
for col in behaviour_cols:
    pct = df[f"had_{col}_before"].mean() * 100
    print(f"  had_{col}_before = 1: {pct:.1f}% of sessions")

The preview above illustrates the temporal shift logic for a single user across
4 sessions. In the first session (2025-06-09), all history flags are 0 by
construction — no prior activity exists. From the second session onwards, flags
update correctly: `had_viewed_product_before` and `had_added_to_cart_before`
become 1 after the first session where those actions were performed, and the
corresponding `n_*_before` counts increment with each additional session.

The aggregate statistics confirm the expected funnel structure in the historical
features: 27.3% of sessions belong to users who had previously viewed a product,
5.5% to users who had previously added to cart, 1.7% to users who had previously
started checkout, and only 0.9% to users who had previously completed a purchase.
Each percentage is a subset of the previous, consistent with the progressive
narrowing of the conversion funnel across sessions.

#### Why User History Features Are Included — and Why They Are Legitimate

The eight features created in this section capture each user's **longitudinal behavioral
history** across sessions prior to the current one:

| Feature | Description |
|---|---|
| `had_viewed_product_before` | 1 if the user viewed a product in any prior session |
| `had_added_to_cart_before` | 1 if the user added to cart in any prior session |
| `had_started_checkout_before` | 1 if the user started checkout in any prior session |
| `had_purchased_before` | 1 if the user completed a purchase in any prior session |
| `n_viewed_product_before` | Count of prior sessions with a product view |
| `n_added_to_cart_before` | Count of prior sessions with a cart addition |
| `n_started_checkout_before` | Count of prior sessions with a checkout start |
| `n_purchased_before` | Count of prior sessions with a completed purchase |

**Why they are legitimate (no leakage):** All features are constructed using a temporal
shift (`shift(fill_value=0).cummax()` for binary flags and `.cumsum()` for counts),
which ensures that only information from sessions *strictly prior* to the current one
is used. For a user's first session, all `had_*` flags are 0 and all `n_*` counts are
0 — no future information is ever included. The user-level train/test split (Section
3.2) further guarantees that the full session history of test users is never visible
to the model during training.

**Why they add value:** Standard session-level features capture only what happened
*within* a single session. In a panel dataset where users visit multiple times,
ignoring prior sessions discards strong signals about user intent and their stage in
the customer lifecycle. Specifically:

- `had_purchased_before` identifies returning buyers who have already overcome the
  trust barrier — it is the strongest individual predictor at the checkout stage
  (LR coefficient: 1.60 in Model 3).
- `n_added_to_cart_before` captures repeated cart-abandonment behavior: users who
  have added items many times across sessions without ever purchasing signal
  systematic indecision, and indeed show a strongly negative coefficient at Models
  2 and 3 (–0.82 and –0.61 respectively).
- `had_started_checkout_before` and `n_started_checkout_before` identify "warm"
  users who came close to converting in a prior session and may respond to a
  targeted nudge.

These features are conceptually equivalent to the **RFM framework** (Recency,
Frequency, Monetary value) commonly used in CRM and direct marketing, applied here
at the funnel-stage level rather than at the transaction level.

### 2.10 Summary of Feature Engineering

This section focused on creating new features and transforming existing ones to enrich the dataset for modeling. Here's a recap of the key changes:

#### Variables Added:

*   **Logarithmic Transformations:** New `log_` prefixed versions were created for several highly skewed numerical variables, including: `log_ga_session_number`, `log_items_viewed`, `log_items_added_to_cart`, `log_checkout_events`, `log_unique_items_viewed`, `log_unique_items_added_to_cart`, `log_unique_items_purchased`, `log_unique_categories_viewed`, `log_cart_value`, `log_avg_viewed_price`, `log_time_view_to_cart`, `log_time_cart_to_checkout`, and `log_time_checkout_to_purchase`.
*   **User ID Categorization:**
    *   `user_id_binary`: A binary indicator (1 if `user_id` is not null, 0 otherwise).
*   **User and Session ID Metrics:**
    *   `user_pseudo_id_unique`: Binary indicating if a `user_pseudo_id` appeared only once.
    *   `session_id_unique`: Binary indicating if a `session_id` appeared only once.
    *   `unique_session`: A composite identifier combining `user_pseudo_id` and `session_id`.
    *   `num_sessions_per_user`: Counts the number of unique sessions for each `user_pseudo_id`.
*   **Ratios of Items:**
    *   `ratio_items_viewed_unique`: `items_viewed` / `unique_items_viewed`.
    *   `ratio_items_added_to_cart_unique`: `items_added_to_cart` / `unique_items_added_to_cart`.
    *   `log_ratio_items_viewed_unique`: Log-transformed version of `ratio_items_viewed_unique`.
    *   `log_ratio_items_added_to_cart_unique`: Log-transformed version of `ratio_items_added_to_cart_unique`.
    *   `ratio_items_viewed>1`: Binary indicating if `ratio_items_viewed_unique` is greater than 1.
    *   `ratio_items_added_to_cart>1`: Binary indicating if `ratio_items_added_to_cart_unique` is greater than 1.
*   **Login Status Handling:**
    *   `logged_in_or_null_dummy`: Binary variable (1 if 'loggedin' or 'null', 0 if 'not logged').
*   **Historical Session Data:**
    *   `user_key`: Combines `user_id` and `user_pseudo_id` to uniquely identify users.
    *   `had_viewed_product_before`, `had_added_to_cart_before`, `had_started_checkout_before`, `had_purchased_before`: Binary flags indicating if the user performed these actions in a previous session.
    *   `n_viewed_product_before`, `n_added_to_cart_before`, `n_started_checkout_before`, `n_purchased_before`: Counts of how many times the user performed these actions before the current session.
*   **Engagement Ratio:**
    `ratio_engagement_duration`: `total_engagement_time` (converted from milliseconds
to seconds by dividing by 1,000) divided by `session_duration_sec`. Represents the
proportion of the session duration spent actively engaged with content.

#### Variables Removed/Modified:

*   **Removed (no longer present in the final DataFrame):**
    *   `first_view_datetime`, `first_cart_datetime`, `first_checkout_datetime`, `purchase_datetime`
    *   `first_view_ts`, `first_cart_ts`, `first_checkout_ts`, `purchase_ts`
    *   `unique_items_purchased` (only its log-transformed version remains)
    *   `total_engagement_time_sec` (intermediate variable)
*   **Modified In-place:**
    *   `login_status`: Null values were replaced with the string 'null' to be treated as a distinct category.
    *   Original columns like `ga_session_number`, `items_viewed`, `items_added_to_cart`, `checkout_events`, `unique_items_viewed`, `unique_items_added_to_cart`, `unique_categories_viewed`, `cart_value`, `avg_viewed_price`, `time_view_to_cart`, `time_cart_to_checkout`, `time_checkout_to_purchase` were retained alongside their `log_` transformed versions.

In [ ]:
# Convert total_engagement_time from milliseconds to seconds
df['total_engagement_time_sec'] = df['total_engagement_time'] / 1000

# Calculate the ratio, handling division by zero and NaN values
df['ratio_engagement_duration'] = df.apply(
    lambda row:
        row['total_engagement_time_sec'] / row['session_duration_sec']
        if row['session_duration_sec'] > 0 and pd.notna(row['total_engagement_time_sec'])
        else 0,
    axis=1
)
df = df.drop(columns=["total_engagement_time_sec"])
print("New variable 'ratio_engagement_duration' created.")
display(df[['total_engagement_time', 'ratio_engagement_duration']].head())

In [ ]:
# Distribution of ratio_engagement_duration
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(df['ratio_engagement_duration'].dropna(), bins=50,
             color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of ratio_engagement_duration')
axes[0].set_xlabel('Ratio (engagement time / session duration)')
axes[0].set_ylabel('Frequency')

# By purchase outcome
df[df['purchased'] == 0]['ratio_engagement_duration'].dropna().hist(
    bins=50, alpha=0.6, color='tomato', label='Not purchased', ax=axes[1])
df[df['purchased'] == 1]['ratio_engagement_duration'].dropna().hist(
    bins=50, alpha=0.6, color='steelblue', label='Purchased', ax=axes[1])
axes[1].set_title('ratio_engagement_duration by Purchase Outcome')
axes[1].set_xlabel('Ratio (engagement time / session duration)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print(df['ratio_engagement_duration'].describe().round(3))
print(f"\nSessions with ratio > 1 (inconsistent): {(df['ratio_engagement_duration'] > 1).sum()}")
print(f"Sessions with ratio = 0: {(df['ratio_engagement_duration'] == 0).sum()}")

The `ratio_engagement_duration` variable measures **engagement intensity** —
the fraction of total session time during which the user was actively interacting
with the site, as tracked by GA4.

**Key statistics:**
- The variable is bounded between 0 and 1 by construction, with no inconsistent
  values (sessions with ratio > 1: 0).
- **43,461 sessions (22.5%) have a ratio of exactly 0**, corresponding to sessions
  where either `session_duration_sec` is zero or `total_engagement_time` is missing.
  These are typically very short bounce sessions with no meaningful engagement.
- The **mean ratio is 0.40** and the **median is 0.34**, indicating that in a
  typical session roughly a third of the total time is spent in active interaction.
- The distribution is **bimodal**: a sharp spike at 0 (bounce/inactive sessions)
  and a broad, relatively flat distribution between 0.6 and 1.0, suggesting two
  distinct user populations — passive browsers and actively engaged users.

**Comparison by purchase outcome** (right panel): the separation between purchasers
and non-purchasers is not visually dramatic because the purchased group is a very
small minority of the total (less than 1%), making it nearly invisible on the shared
frequency scale. However, the models in Sections 3 and 4 confirm that
`ratio_engagement_duration` has a consistently positive and significant effect on
funnel progression at the first two stages, capturing engagement intensity as a
dimension distinct from raw session duration.

### 2.9 Correlation Matrix and Heatmap

In this section, we will present four different correlation matrices and heatmaps to analyze the relationships between numerical features at various stages of the user funnel:

1.  **Correlation matrix with all observations:** This provides an overview of feature correlations across the entire dataset.
2.  **Correlation Matrix and Heatmap for 'Added to Cart' Sessions:** This focuses on sessions where users have added at least one item to their cart, highlighting correlations relevant to this stage.
3.  **Correlation Matrix and Heatmap for 'Checkout Started' Sessions:** This analyzes correlations for sessions where users initiated the checkout process.
4.  **Correlation Matrix and Heatmap for 'Purchased' Sessions:** This examines correlations for sessions that resulted in a successful purchase, providing insights into conversion drivers.

#### 2.9.1 Correlation Matrix with all observations

This initial heatmap provides an overview of the linear relationships between all numerical features across the entire dataset. It helps identify features that are strongly correlated with each other, potentially indicating redundancy, or with key behavioral indicators like `log_total_engagement_time` and `log_session_duration_sec`, which are expected to show strong internal correlations. For example, `log_total_engagement_time` and `log_session_duration_sec` generally exhibit a strong positive correlation, as users who spend more time on the site tend to be more engaged overall.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create df_base here as a snapshot of the fully engineered df
df_base = df.copy()

# Identify non-binary numerical columns for correlation analysis
# Exclude identifiers and columns that are already binary or handled as categories
# Also exclude columns with too many NaNs that would make correlation computation difficult

# Start with numerical columns defined during feature engineering, or infer new ones
# Ensure `df` is the current working DataFrame after all feature engineering steps

# Re-define numerical features that are not explicitly binary from the main df
non_binary_numerical_cols = [
    'log_ga_session_number',
    'log_items_viewed',
    'log_items_added_to_cart',
    'log_checkout_events',
    'log_unique_items_viewed',
    'log_unique_items_added_to_cart',
    'log_unique_items_purchased',
    'log_unique_categories_viewed',
    'log_cart_value',
    'log_avg_viewed_price',
    'log_total_engagement_time',
    'log_session_duration_sec',
    'log_time_view_to_cart',
    'log_time_cart_to_checkout',
    'log_time_checkout_to_purchase',
    'num_sessions_per_user',
    'log_ratio_items_viewed_unique',
    'log_ratio_items_added_to_cart_unique',
    'ratio_engagement_duration'
]

# Filter to ensure columns actually exist in the DataFrame
current_numerical_cols = [col for col in non_binary_numerical_cols if col in df_base.columns]

# Calculate the correlation matrix
correlation_matrix = df_base[current_numerical_cols].corr()

# Plot the heatmap
plt.figure(figsize=(20, 16)) # Increased figure size for better readability
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Numerical Variables', fontsize=18)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

> **Note on potentially leakage-prone variables:** The correlation matrix includes
> `log_time_checkout_to_purchase` and `log_unique_items_purchased` for completeness
> of the exploratory analysis. Both variables are structurally linked to the target:
> `log_time_checkout_to_purchase` only exists for sessions that resulted in a purchase,
> and `log_unique_items_purchased` is zero for all non-converting sessions. They are
> included here solely to observe their correlation structure with other features.
> **Neither variable is used as a predictor in any of the models in Sections 3 and 4.**

#### 2.9.2 Correlation Matrix and Heatmap for 'Added to Cart' Sessions

This section presents the correlation matrix and heatmap, specifically filtered for sessions where the user has added at least one item to the cart (`added_to_cart == 1`). This allows us to observe the relationships between numerical features that are particularly relevant to the 'Add to Cart' stage and subsequent funnel progression. Key observations typically include strong positive correlations between `log_total_engagement_time`, `log_session_duration_sec`, and various `log_items_viewed` metrics, suggesting that engaged users who explore more products are more likely to add items to their cart. Conversely, `log_avg_viewed_price` and `log_unique_items_viewed` often show negative relationships, indicating that users viewing higher-priced items or a greater variety of unique items (potentially suggesting less focus) are less likely to add items to the cart.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Identify non-binary numerical columns for correlation analysis
# Using the same list as before, ensuring it is relevant for df_m2

current_numerical_cols_m2 = [
    'log_ga_session_number',
    'log_items_viewed',
    'log_items_added_to_cart',
    'log_checkout_events',
    'log_unique_items_viewed',
    'log_unique_items_added_to_cart',
    'log_unique_items_purchased',
    'log_unique_categories_viewed',
    'log_cart_value',
    'log_avg_viewed_price',
    'log_total_engagement_time', # Added this back
    'log_session_duration_sec',  # Added this back
    'log_time_view_to_cart',
    'log_time_cart_to_checkout',
    'log_time_checkout_to_purchase',
    'num_sessions_per_user',
    'log_ratio_items_viewed_unique',
    'log_ratio_items_added_to_cart_unique',
    'ratio_engagement_duration'
]

# Define df_m2 for this section
df_m2 = df_base[df_base['added_to_cart'] == 1].copy()

# Filter to ensure columns actually exist in the DataFrame
filtered_numerical_cols_m2 = [col for col in current_numerical_cols_m2 if col in df_m2.columns]

# Calculate the correlation matrix for the filtered DataFrame (df_m2)
correlation_matrix_m2 = df_m2[filtered_numerical_cols_m2].corr()

# Plot the heatmap
plt.figure(figsize=(20, 16)) # Increased figure size for better readability
sns.heatmap(correlation_matrix_m2, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Numerical Variables (Sessions with Added to Cart)', fontsize=18)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

> **Note:** As in Section 2.8.1, `log_time_checkout_to_purchase` and
> `log_unique_items_purchased` are included in this heatmap for exploratory purposes
> only and are excluded from all predictive models to avoid target leakage.

#### 2.9.3 Correlation Matrix and Heatmap for 'Checkout Started' Sessions

This section analyzes correlations for sessions where users initiated the checkout process (`started_checkout == 1`). This allows us to observe the relationships between numerical features that are particularly relevant to the 'Checkout Started' stage and the final purchase. At this stage, `log_total_engagement_time` and `log_session_duration_sec` typically remain strong positive correlates with progression. Features like `log_cart_value` and `log_time_view_to_cart` become more prominent, with `log_cart_value` often positively correlated and `log_time_view_to_cart` (shorter times indicating quicker decisions) negatively correlated with proceeding to checkout.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define the numerical features to include in the correlation matrix for df_m3
# These are selected from md3_features, excluding binary/dummy variables
numerical_features_for_corr_m3 = [
    'log_ga_session_number',
    'log_items_viewed',
    'log_unique_items_viewed',
    'log_unique_categories_viewed',
    'log_avg_viewed_price',
    'log_total_engagement_time',
    'log_session_duration_sec',
    'ratio_engagement_duration',
    'log_ratio_items_viewed_unique',
    'log_items_added_to_cart',
    'log_unique_items_added_to_cart',
    'log_cart_value',
    'log_time_view_to_cart',
    'log_ratio_items_added_to_cart_unique',
    'log_checkout_events',
    'log_time_cart_to_checkout',
    'num_sessions_per_user',
    'n_viewed_product_before',
    'n_added_to_cart_before',
    'n_started_checkout_before',
    'n_purchased_before'
]

# Define df_m3 for this section
df_m3 = df_base[df_base['started_checkout'] == 1].copy()

# Filter to ensure columns actually exist in df_m3
filtered_numerical_cols_m3 = [col for col in numerical_features_for_corr_m3 if col in df_m3.columns]

# Calculate the correlation matrix for the filtered DataFrame (df_m3)
correlation_matrix_m3 = df_m3[filtered_numerical_cols_m3].corr()

# Plot the heatmap
plt.figure(figsize=(20, 16)) # Increased figure size for better readability
sns.heatmap(correlation_matrix_m3, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Numerical Variables (Sessions with Checkout Started)', fontsize=18)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

#### 2.9.4 Correlation Matrix and Heatmap for 'Purchased' Sessions

This final section presents the correlation matrix and heatmap, specifically filtered for sessions where the user has completed a purchase (`purchased == 1`). This is the ultimate stage of the funnel, and this visualization will highlight the relationships between numerical features that are most indicative of a successful conversion. While correlations are generally weaker at this stage due to the highly self-selected group of converters, `log_session_duration_sec` and `log_total_engagement_time` remain important. Notably, `log_time_cart_to_checkout` can emerge as a significant negative correlate (longer times indicating hesitation), while `log_avg_viewed_price` often shows a negative relationship with purchase completion, reflecting price sensitivity at the final step.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define the numerical features to include in the correlation matrix for df_m4 (purchased sessions)
# These should be all relevant numerical features that are available at the point of purchase

md4_features = [
    'log_ga_session_number',
    'log_items_viewed',
    'log_unique_items_viewed',
    'log_unique_categories_viewed',
    'log_avg_viewed_price',
    'log_total_engagement_time',
    'log_session_duration_sec',
    'ratio_engagement_duration',
    'log_ratio_items_viewed_unique',
    'ratio_items_viewed>1',
    'log_items_added_to_cart',
    'log_unique_items_added_to_cart',
    'log_cart_value',
    'log_time_view_to_cart',
    'log_ratio_items_added_to_cart_unique',
    'ratio_items_added_to_cart>1',
    'log_checkout_events',
    'log_time_cart_to_checkout',
    'log_time_checkout_to_purchase',
    'num_sessions_per_user',
    'n_viewed_product_before',
    'n_added_to_cart_before',
    'n_started_checkout_before',
    'n_purchased_before'
]

# Filter the base DataFrame for purchased sessions
df_m4 = df_base[df_base['purchased'] == 1].copy()

# Filter to ensure columns actually exist in df_m4
filtered_numerical_cols_m4 = [col for col in md4_features if col in df_m4.columns]

# Calculate the correlation matrix for the filtered DataFrame (df_m4)
correlation_matrix_m4 = df_m4[filtered_numerical_cols_m4].corr()

# Plot the heatmap
plt.figure(figsize=(20, 16)) # Increased figure size for better readability
sns.heatmap(correlation_matrix_m4, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Numerical Variables (Sessions with Purchase)', fontsize=18)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### 2.10 Summary of Feature Engineering

This section focused on creating new features and transforming existing ones to enrich the dataset for modeling. Here's a recap of the key changes:

#### Variables Added:

*   **Logarithmic Transformations:** New `log_` prefixed versions were created for several highly skewed numerical variables, including: `log_ga_session_number`, `log_items_viewed`, `log_items_added_to_cart`, `log_checkout_events`, `log_unique_items_viewed`, `log_unique_items_added_to_cart`, `log_unique_items_purchased`, `log_unique_categories_viewed`, `log_cart_value`, `log_avg_viewed_price`, `log_time_view_to_cart`, `log_time_cart_to_checkout`, and `log_time_checkout_to_purchase`.
*   **User ID Categorization:**
    *   `user_id_binary`: A binary indicator (1 if `user_id` is not null, 0 otherwise).
*   **User and Session ID Metrics:**
    *   `user_pseudo_id_unique`: Binary indicating if a `user_pseudo_id` appeared only once.
    *   `session_id_unique`: Binary indicating if a `session_id` appeared only once.
    *   `unique_session`: A composite identifier combining `user_pseudo_id` and `session_id`.
    *   `num_sessions_per_user`: Counts the number of unique sessions for each `user_pseudo_id`.
*   **Ratios of Items:**
    *   `ratio_items_viewed_unique`: `items_viewed` / `unique_items_viewed`.
    *   `ratio_items_added_to_cart_unique`: `items_added_to_cart` / `unique_items_added_to_cart`.
    *   `log_ratio_items_viewed_unique`: Log-transformed version of `ratio_items_viewed_unique`.
    *   `log_ratio_items_added_to_cart_unique`: Log-transformed version of `ratio_items_added_to_cart_unique`.
    *   `ratio_items_viewed>1`: Binary indicating if `ratio_items_viewed_unique` is greater than 1.
    *   `ratio_items_added_to_cart>1`: Binary indicating if `ratio_items_added_to_cart_unique` is greater than 1.
*   **Login Status Handling:**
    *   `logged_in_or_null_dummy`: Binary variable (1 if 'loggedin' or 'null', 0 if 'not logged').
*   **Historical Session Data:**
    *   `user_key`: Combines `user_id` and `user_pseudo_id` to uniquely identify users.
    *   `had_viewed_product_before`, `had_added_to_cart_before`, `had_started_checkout_before`, `had_purchased_before`: Binary flags indicating if the user performed these actions in a previous session.
    *   `n_viewed_product_before`, `n_added_to_cart_before`, `n_started_checkout_before`, `n_purchased_before`: Counts of how many times the user performed these actions before the current session.
*   **Engagement Ratio:**
    `ratio_engagement_duration`: `total_engagement_time` (converted from milliseconds
to seconds by dividing by 1,000) divided by `session_duration_sec`. Represents the
proportion of the session duration spent actively engaged with content.

#### Variables Removed/Modified:

*   **Removed (no longer present in the final DataFrame):**
    *   `first_view_datetime`, `first_cart_datetime`, `first_checkout_datetime`, `purchase_datetime`
    *   `first_view_ts`, `first_cart_ts`, `first_checkout_ts`, `purchase_ts`
    *   `unique_items_purchased` (only its log-transformed version remains)
    *   `total_engagement_time_sec` (intermediate variable)
*   **Modified In-place:**
    *   `login_status`: Null values were replaced with the string 'null' to be treated as a distinct category.
    *   Original columns like `ga_session_number`, `items_viewed`, `items_added_to_cart`, `checkout_events`, `unique_items_viewed`, `unique_items_added_to_cart`, `unique_categories_viewed`, `cart_value`, `avg_viewed_price`, `time_view_to_cart`, `time_cart_to_checkout`, `time_checkout_to_purchase` were retained alongside their `log_` transformed versions.

## 3 Funnel-Based Logistic Regression (Chain Rule Decomposition)

In this section, we move from data preparation and feature engineering to model building. Our primary objective is to analyze and predict customer purchase behavior within an e-commerce funnel. Instead of modeling purchase as a simple binary outcome (0/1) directly, which can be problematic due to severe class imbalance and loss of stage-specific insights, we adopt a more nuanced approach.

We decompose the overall probability of purchase using the **chain rule of probability**:

$$P(\text{purchase}) = P(\text{add to cart} \mid \text{viewed product}) \times P(\text{checkout} \mid \text{added to cart}) \times P(\text{purchase} \mid \text{started checkout})$$

Each conditional probability is estimated with a separate Logistic Regression model. This approach:
- **Preserves the funnel structure** rather than collapsing all non-purchasers into a single category. This avoids the loss of information that the simpler strategy would have created, because we would have dropped more than 95% of the dataset.
- **Avoids cross-stage leakage**: each model only uses features observable at that specific stage
- **Produces stage-specific insights** that are directly actionable for Versace (e.g. where exactly users drop off and why). In addition, the coefficient coming from the regression can be used to see how the probability of moving from one stage to the following one changes across stages, and we can retrieve the importance of each variable in each step of the funnel.  
- **Creates more balanced classes**: just a very tiny percentage of users actually purchased at least an item (less than 1%). Modeling purchase vs non-purchase would create imbalanced classes that a logistic regression would not address well and we would rely on undersampling techniques. Instead, this separation overcomes this problem and allows the use of this machine learning method.

### 3.1 Prepare Base Dataframe and Encode Categorical Features

We start from the cleaned dataframe produced in the preprocessing section. We one-hot encode `time_of_day_segment` and to be kept throughout the different models. From the first variable, we get a set of dummy variables for each hour of the day, whereas from the second one we obtain a binary indicator of each segment of the day (morning, afternoon, evening and night).

In [ ]:
pd.set_option('mode.chained_assignment', None)

df_base = df.copy()

# One-hot encode 'first_view_hour', 'time_of_day_segment', and 'login_status'
df_base = pd.get_dummies(df_base,
                         columns=['time_of_day_segment',
                                  'login_status'
                                  ],
                         drop_first=True)

# Define dummy feature lists
time_of_day_segment_dummies = [c for c in df_base.columns if c.startswith('time_of_day_segment_')]
login_dummies = [c for c in df_base.columns if c.startswith('login_status_')]

### 3.2 User-Aware Train/Test Split

A standard random split would allow the same user to appear in both train and test sets, artificially inflating performance metrics — a form of panel data leakage. Instead, we split at the **user level**: all sessions belonging to a given `user_pseudo_id` are assigned entirely to either train or test, never both.

In [ ]:
unique_users = df_base['user_pseudo_id'].unique()

users_train, users_test = train_test_split(
    unique_users, test_size=0.3, random_state=42
)

train_mask = df_base['user_pseudo_id'].isin(users_train)
test_mask  = df_base['user_pseudo_id'].isin(users_test)

print(f"Total unique users: {len(unique_users)}")
print(f"Users in train: {len(users_train)} ({len(users_train)/len(unique_users)*100:.1f}%)")
print(f"Users in test:  {len(users_test)} ({len(users_test)/len(unique_users)*100:.1f}%)")
print(f"Train rows: {train_mask.sum()} | Test rows: {test_mask.sum()}")

### 3.3 Helper Function: Fit and Evaluate a Logistic Regression Model

To avoid code repetition across the three funnel models, we define a reusable function that fits a logistic regression, prints the classification report and ROC AUC score, and produces two plots: the ROC curve and the top 15 features ranked by absolute coefficient value. Blue bars indicate features that increase the probability of progression; red bars indicate features that decrease it.

We use `class_weight='balanced'` to handle class imbalance, which is severe especially at the view → cart transition where only a small fraction of sessions result in a cart addition.

In [ ]:
def fit_evaluate_lr(X_train, X_test, y_train, y_test, model_name):
    """
    Fits a logistic regression with balanced class weights,
    evaluates performance, and plots ROC curve + feature importance.

    Returns: model, scaler, results dict
    """
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    model = LogisticRegression(
        class_weight='balanced',
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train_sc, y_train)

    y_pred       = model.predict(X_test_sc)
    y_pred_proba = model.predict_proba(X_test_sc)[:, 1]

    auc = roc_auc_score(y_test, y_pred_proba)

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  Positive class rate (test): {y_test.mean()*100:.2f}%")
    print(f"  ROC AUC Score: {auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'ROC Curve — {model_name}')
    axes[0].legend()

    coef_df = pd.DataFrame({
        'feature':     X_train.columns,
        'coefficient': model.coef_[0]
    }).assign(abs_coef=lambda d: d['coefficient'].abs()) \
      .sort_values('abs_coef', ascending=False) \
      .head(15)

    colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['coefficient']]
    axes[1].barh(coef_df['feature'], coef_df['coefficient'],
                 color=colors, edgecolor='white')
    axes[1].axvline(0, color='black', lw=0.8)
    axes[1].set_xlabel('Coefficient')
    axes[1].set_title(f'Top 15 Features — {model_name}')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    return model, scaler, {
        'model_name':   model_name,
        'auc':          auc,
        'coef_df':      coef_df,
        'y_pred_proba': y_pred_proba,
        'y_test':       y_test
    }

A Note on Multicollinearity: `total_engagement_time`, `session_duration_sec`, and `ratio_engagement_duration`

To address potential multicollinearity among these three highly correlated variables (one is even a linear combination of the others), we opted to exclude `total_engagement_time` from our models. This decision allows for a clearer and less redundant understanding of user behavior by focusing on two distinct, yet complementary, measures:

*   **`session_duration_sec`**: This variable captures the *overall length* of a user's session.
*   **`ratio_engagement_duration`**: This variable quantifies the *proportion of active engagement* within that total session duration.

By separating these metrics, we aim to provide more precise insights into how both the total time spent and the intensity of engagement influence user behavior within the funnel.

### 3.4 Logistic Regression Model

#### 3.4.1 Model 1: P(Add to Cart | Viewed Product) - Using `time_of_day_segment`

This is a second version of Model 1, estimating the probability that a user adds at least one item to the cart, given that they viewed at least one product. Similar to the previous version, the training subset is restricted to sessions where `viewed_product == 1`.

This model includes the `time_of_day_segment` dummy variables. This is because the individual `first_view_hour` dummies were found to be largely insignificant in the previous model, while `time_of_day_segment` provides a broader, potentially more robust, temporal context without introducing multicollinearity with other hour-based features.

Only features observable **at the viewing stage** are included.

In [ ]:
import statsmodels.api as sm

df_m1 = df_base[df_base['viewed_product'] == 1].copy()
md1_features_v2 = (
    ['log_ga_session_number',
     'items_viewed',
     'unique_items_viewed',
     'log_unique_categories_viewed',
     'log_avg_viewed_price',
     'session_duration_sec',
     'ratio_engagement_duration',
     'ratio_items_viewed>1',
     'num_sessions_per_user',
     'user_id_binary',
     'logged_in_or_null_dummy',
     "n_viewed_product_before",
     "n_added_to_cart_before"]
    + time_of_day_segment_dummies
)

target_1 = 'added_to_cart'

df_m1_v2 = df_base[df_base['viewed_product'] == 1].copy()

print(f"[Model 1 - V2] Sessions with product views: {len(df_m1_v2)}")
print(f"[Model 1 - V2] Positive rate (added to cart): {df_m1_v2[target_1].mean()*100:.2f}%")

X_train_1_v2 = df_m1_v2.loc[df_m1_v2['user_pseudo_id'].isin(users_train), md1_features_v2]
X_test_1_v2  = df_m1_v2.loc[df_m1_v2['user_pseudo_id'].isin(users_test),  md1_features_v2]
y_train_1_v2 = df_m1_v2.loc[df_m1_v2['user_pseudo_id'].isin(users_train), target_1]
y_test_1_v2  = df_m1_v2.loc[df_m1_v2['user_pseudo_id'].isin(users_test),  target_1]

for col in md1_features_v2:
    if X_train_1_v2[col].isnull().any():
        med = X_train_1_v2[col].median()
        X_train_1_v2[col] = X_train_1_v2[col].fillna(med)
        X_test_1_v2[col]  = X_test_1_v2[col].fillna(med)

model_1_v2, scaler_1_v2, results_1_v2 = fit_evaluate_lr(
    X_train_1_v2, X_test_1_v2, y_train_1_v2, y_test_1_v2,
    model_name="Model 1 - V2: P(Add to Cart | Viewed Product) - Time of Day Segment"
)

In [ ]:
X_train_sm_v2 = sm.add_constant(X_train_1_v2).astype(float)
X_test_sm_v2 = sm.add_constant(X_test_1_v2).astype(float)

logit_model_v2 = sm.Logit(y_train_1_v2, X_train_sm_v2)
result_v2 = logit_model_v2.fit()

print(result_v2.summary())

**Model 1 — Results**

**1. Model Performance:**
- **ROC AUC: 0.8243** — marginally higher than Model 1.1 (0.8395) despite using
  far fewer parameters (17 vs 54 degrees of freedom in `statsmodels`). This confirms that replacing
  the granular hour and date dummies with `time_of_day_segment` produces a more
  parsimonious model without sacrificing predictive power.
- **Positive class rate (test): 5.86%** — (from the `fit_evaluate_lr` output, which remains consistent).
- **Recall for class 1: 0.68**, **Precision for class 1: 0.17** — (from the `fit_evaluate_lr` output, which remains consistent) The low precision is expected and acceptable given the use of `class_weight='balanced'` on a heavily imbalanced target.
- **Pseudo R-squared: 0.2064** — lower than Model 1.1 (0.2384), which
  is expected given the large reduction in the number of parameters. The drop is
  negligible and consistent with the near-zero contribution of the removed dummies.

**2. Key Significant Predictors (from statsmodels output):**

Positive influence on P(Add to Cart | Viewed Product):
- `session_duration_sec` (coef: 0.0001, p<0.001) and `ratio_engagement_duration` (coef: 1.2979, p<0.001) — active engagement and session duration are strong signals of cart intent
- `user_id_binary` (coef: 2.1477, p<0.001) — identified users are far more likely to add to cart
- `ratio_items_viewed>1` (coef: 1.2214, p<0.001) — repeated views of the same
  items signal focused interest
- `log_unique_categories_viewed` (coef: 1.3774, p<0.001) — browsing across categories
  increases add-to-cart probability.
- `items_viewed` (coef: 0.1693, p<0.001) — viewing more items increases add-to-cart probability.
- `n_added_to_cart_before` (coef: 0.3346, p<0.001) — prior cart additions are a strong
  positive predictor of current cart addition.
- `logged_in_or_null_dummy` (coef: 1.6803, p<0.001) - Identified users show higher intent.
- `n_viewed_product_before` (coef: -0.0784, p<0.001) - More previous product views might indicate indecision.
- `log_ga_session_number` (coef: 0.0814, p=0.009) - A higher session number is associated with increased cart addition probability.

Negative influence:
- `log_avg_viewed_price` (coef: -0.5411, p<0.001) — users viewing higher-priced items
  are less likely to add to cart, reflecting price sensitivity at the top of
  the funnel.
- `unique_items_viewed` (coef: -0.2601, p<0.001) — dispersive browsing reduces cart
  addition probability.

Non-significant predictors (p > 0.05):
- `num_sessions_per_user` (coef: 0.0042, p=0.222)

**3. `time_of_day_segment` Variables:**
- `time_of_day_segment_evening` (coef: -0.1668, p=0.012) - Significantly lower probability of adding to cart in the evening.
- `time_of_day_segment_morning` (coef: 0.0219, p=0.722)
- `time_of_day_segment_night` (coef: -0.0130, p=0.904)

  Only `time_of_day_segment_evening` is statistically significant in this model specification. This suggests that, once other behavioral factors are accounted for, sessions in the evening have a lower probability of adding to cart.

**Conclusion:**
Model 1.2 remains the preferred specification for the View → Add to Cart stage. It achieves a very similar AUC to Model 1.1 with significantly fewer parameters, confirming that `time_of_day_segment` is a more stable and parsimonious temporal control than granular hour or date dummies. The core findings regarding engagement, identified users, and focused browsing remain robust.

#### 3.4.2 Model 2.1: P(Start Checkout | Added to Cart)

The second model estimates the probability that a user proceeds to checkout, given that they added at least one item to the cart. The training subset is restricted to sessions where `added_to_cart == 1`.

Cart-specific features such as `cart_value`and `time_view_to_cart` are now available and included, as they are observable once the user has reached this stage.

In [ ]:
md2_features = (
    ['log_ga_session_number',
     'items_viewed',
     'unique_items_viewed',
     'ratio_items_viewed>1',
     'log_unique_categories_viewed',
     'log_avg_viewed_price',
     'session_duration_sec',
     'ratio_engagement_duration',
     'items_added_to_cart',
     'unique_items_added_to_cart',
     'ratio_items_added_to_cart>1',
     'log_cart_value',
     'log_time_view_to_cart',
     'num_sessions_per_user',
     'user_id_binary',
     'logged_in_or_null_dummy',
     "n_viewed_product_before",
     "n_added_to_cart_before",
     "n_started_checkout_before"]
    + [c for c in df_base.columns if c.startswith('time_of_day_segment_')]
)

target_2 = 'started_checkout'

df_m2 = df_base[df_base['added_to_cart'] == 1].copy()

print(f"[Model 2] Sessions with cart additions: {len(df_m2)}")
print(f"[Model 2] Positive rate (started checkout): {df_m2[target_2].mean()*100:.2f}%")

X_train_2 = df_m2.loc[df_m2['user_pseudo_id'].isin(users_train), md2_features]
X_test_2  = df_m2.loc[df_m2['user_pseudo_id'].isin(users_test),  md2_features]
y_train_2 = df_m2.loc[df_m2['user_pseudo_id'].isin(users_train), target_2]
y_test_2  = df_m2.loc[df_m2['user_pseudo_id'].isin(users_test),  target_2]

for col in md2_features:
    if X_train_2[col].isnull().any():
        med = X_train_2[col].median()
        X_train_2[col] = X_train_2[col].fillna(med)
        X_test_2[col]  = X_test_2[col].fillna(med)

model_2, scaler_2, results_2 = fit_evaluate_lr(
    X_train_2, X_test_2, y_train_2, y_test_2,
    model_name="Model 2: P(Start Checkout | Added to Cart)"
)

In [ ]:
X_train_sm = sm.add_constant(X_train_2).astype(float)
X_test_sm = sm.add_constant(X_test_2).astype(float)

logit_model = sm.Logit(y_train_2, X_train_sm)
result = logit_model.fit()

print(result.summary())

The logistic regression *does* converge, but the statsmodels output below shows that the `first_cart_date` dummy variables are not statistically significant and introduce numerical instability due to near-collinearity. Therefore, we still choose to re-estimate the model without these dummies in Model 2.2 for a more parsimonious and stable specification.

The statsmodels output reveals that all `first_cart_date` dummies show p-values
well above 0.05 and standard errors around 1.63 — indicating near-collinearity
and no statistically significant effect of the specific cart date on checkout
progression. While the model *did* converge (Pseudo R-squared: 0.2486), including
these 61 parameters adds noise without improving model fit (compared to Model 2.2 below).

This specification is reported for completeness and methodological transparency.
The `first_cart_date` dummies are removed in Model 2.2, which represents the
final stable specification used in all subsequent analyses.

#### 3.4.3 Model 3: P(Purchase | Started Checkout)

The third model estimates the probability of completing a purchase, given that the user started the checkout process. The training subset is restricted to sessions where `started_checkout == 1`.

Checkout-specific features such as `checkout_events`, `time_cart_to_checkout`, and `first_checkout_hour` are now included. Note that `time_checkout_to_purchase` is **deliberately excluded**: this variable only exists after a purchase occurs, so including it would constitute direct target leakage.

Here we excluded the first_checkout date from the model because it failed to converge. We keep the simpler spcification.


In [ ]:
md3_features = (
    ['log_ga_session_number',
     'items_viewed',
     'unique_items_viewed',
     'ratio_items_viewed>1',
     'log_unique_categories_viewed',
     'log_avg_viewed_price',
     'session_duration_sec',
     'ratio_engagement_duration',
     'ratio_items_added_to_cart>1',
     'items_added_to_cart',
     'unique_items_added_to_cart',
     'log_cart_value',
     'log_checkout_events',
     'log_time_view_to_cart',
     'log_time_cart_to_checkout',
     'num_sessions_per_user',
     'user_id_binary',
     'logged_in_or_null_dummy',
     "n_viewed_product_before",
     "n_added_to_cart_before",
     "n_started_checkout_before", # Removed n_viewed_product_before, n_added_to_cart_before
     "n_purchased_before"] # Kept n_started_checkout_before and n_purchased_before as they are for later stages
    + time_of_day_segment_dummies # Uses time_of_day_segment_dummies for consistency
    )

target_3 = 'purchased'

df_m3 = df_base[df_base['started_checkout'] == 1].copy()

print(f"[Model 3] Sessions with checkout started: {len(df_m3)}")
print(f"[Model 3] Positive rate (purchased): {df_m3[target_3].mean()*100:.2f}%")

X_train_3 = df_m3.loc[df_m3['user_pseudo_id'].isin(users_train), md3_features]
X_test_3  = df_m3.loc[df_m3['user_pseudo_id'].isin(users_test),  md3_features]
y_train_3 = df_m3.loc[df_m3['user_pseudo_id'].isin(users_train), target_3]
y_test_3  = df_m3.loc[df_m3['user_pseudo_id'].isin(users_test),  target_3]

for col in md3_features:
    if X_train_3[col].isnull().any():
        med = X_train_3[col].median()
        X_train_3[col] = X_train_3[col].fillna(med)
        X_test_3[col]  = X_test_3[col].fillna(med)

model_3, scaler_3, results_3 = fit_evaluate_lr(
    X_train_3, X_test_3, y_train_3, y_test_3,
    model_name="Model 3: P(Purchase | Started Checkout)"
)

In [ ]:
X_train_sm = sm.add_constant(X_train_3).astype(float)
X_test_sm = sm.add_constant(X_test_3).astype(float)

logit_model = sm.Logit(y_train_3, X_train_sm)
result = logit_model.fit()

print(result.summary())

**Model 3 — Results**

**1. Model Performance:**
- **ROC AUC: 0.7034** — the lowest AUC across the three stages. Users who reach checkout have already self-selected as high-intent shoppers, making it inherently harder to discriminate between those who complete the purchase and those who abandon. The AUC is nonetheless well above random (0.5), confirming that behavioral data retains some predictive power even at this final stage.
- **Positive class rate (test): 45.72%** — nearly balanced classes, meaning
  the model is not penalized by imbalance at this stage.
- **Recall for class 1: 0.67**, **Precision for class 1: 0.63** — good performance for a nearly balanced class distribution.
- **Pseudo R-squared: 0.1356** — significantly lower than the previous iteration (0.3037), indicating that the current variable set has less explanatory power at this stage.
- **N test sessions: 269**.

**2. Key Significant Predictors (from statsmodels output):**

Positive influence on P(Purchase | Started Checkout):
- `log_ga_session_number` (coef: 0.3036, p=0.015) - A higher session number is associated with increased purchase probability at this stage.
- `log_time_cart_to_checkout` (coef: 0.3266, p<0.001) - A longer time between adding to cart and starting checkout increases the probability of purchase.
- `user_id_binary` (coef: 0.5627, p=0.025) - Identified users are more likely to purchase.
- `logged_in_or_null_dummy` (coef: 0.9079, p=0.011) - Logged in or null status increases purchase probability.

Negative influence:
- `log_unique_categories_viewed` (coef: -1.2179, p=0.015) - Browsing across many unique categories decreases purchase probability.
- `num_sessions_per_user` (coef: -0.0315, p=0.013) - A higher number of sessions for a user decreases the purchase probability.
- `time_of_day_segment_evening` (coef: -0.6674, p=0.007) - Evening sessions have a lower purchase probability.
- `time_of_day_segment_night` (coef: -1.1392, p=0.017) - Night sessions have a lower purchase probability.


**3. Non-significant predictors (p > 0.05):**
`items_viewed`, `unique_items_viewed`, `ratio_items_viewed>1`, `log_avg_viewed_price`, `session_duration_sec`, `ratio_engagement_duration`, `ratio_items_added_to_cart>1`, `items_added_to_cart`, `unique_items_added_to_cart`, `log_cart_value`, `log_checkout_events`, `log_time_view_to_cart`, `n_viewed_product_before`, `n_added_to_cart_before`, `n_started_checkout_before`, `n_purchased_before`, `time_of_day_segment_morning`.

**Conclusion:**
The lower AUC of Model 3 (0.7034) is still informative: it confirms that pre-checkout behavioral data does not fully explain checkout abandonment. However, the slightly lower Pseudo R-squared indicates that further investigation is needed to capture the nuances of purchase behavior at this final stage. The actionable implication for Versace is still to instrument the checkout flow with more granular GA4 events and address identified friction points. The impact of `num_sessions_per_user`, `time_of_day_segment_evening`, and `time_of_day_segment_night` also provides valuable insights for targeted interventions.

### 3.5 Chain Rule: Computing the Joint Probability of Purchase

Having estimated the three conditional probabilities, we now apply the chain rule to compute the **joint probability of purchase** for every session in the test set that included at least one product view:

$$P(\text{purchase}) = P(\text{add to cart} \mid \text{viewed}) \times P(\text{checkout} \mid \text{added to cart}) \times P(\text{purchase} \mid \text{checkout started})$$

Sessions that did not reach a given stage receive a probability of 0 for that stage and all subsequent ones, as no progression is possible.

In [ ]:
# Define feature lists (copied from previous cells to make this cell more self-contained)
# Make sure to run the individual model training cells (Model 1, Model 2, Model 3)
# just before this cell to ensure `model_X` and `scaler_X` objects are defined.

md1_features_v2 = (
    ['log_ga_session_number',
     'items_viewed',
     'unique_items_viewed',
     'log_unique_categories_viewed',
     'log_avg_viewed_price',
     'session_duration_sec',
     'ratio_engagement_duration',
     'ratio_items_viewed>1',
     'num_sessions_per_user',
     'user_id_binary',
     'logged_in_or_null_dummy',
     "n_viewed_product_before",
     "n_added_to_cart_before"]
    + [c for c in df_base.columns if c.startswith('time_of_day_segment_')]
)

md2_features = (
    ['log_ga_session_number',
     'items_viewed',
     'unique_items_viewed',
     'ratio_items_viewed>1',
     'log_unique_categories_viewed',
     'log_avg_viewed_price',
     'session_duration_sec',
     'ratio_engagement_duration',
     'items_added_to_cart',
     'unique_items_added_to_cart',
     'ratio_items_added_to_cart>1',
     'log_cart_value',
     'log_time_view_to_cart',
     'num_sessions_per_user',
     'user_id_binary',
     'logged_in_or_null_dummy',
     "n_viewed_product_before",
     "n_added_to_cart_before",
     "n_started_checkout_before"]
    + [c for c in df_base.columns if c.startswith('time_of_day_segment_')]
)

md3_features = (
    ['log_ga_session_number',
     'items_viewed',
     'unique_items_viewed',
     'ratio_items_viewed>1',
     'log_unique_categories_viewed',
     'log_avg_viewed_price',
     'session_duration_sec',
     'ratio_engagement_duration',
     'ratio_items_added_to_cart>1',
     'items_added_to_cart',
     'unique_items_added_to_cart',
     'log_cart_value',
     'log_checkout_events',
     'log_time_view_to_cart',
     'log_time_cart_to_checkout',
     'num_sessions_per_user',
     'user_id_binary',
     'logged_in_or_null_dummy',
     "n_viewed_product_before",
     "n_added_to_cart_before",
     "n_started_checkout_before",
     "n_purchased_before"]
    + [c for c in df_base.columns if c.startswith('time_of_day_segment_')]
)

# Ensure users_test and users_train are defined from the user-aware split cell
# Ensure df_base is defined from the Prepare Base Dataframe cell

df_chain = df_base[
    (df_base['viewed_product'] == 1) &
    (df_base['user_pseudo_id'].isin(users_test))
].copy()

X_chain_1 = df_chain[md1_features_v2].copy()
for col in md1_features_v2:
    if X_chain_1[col].isnull().any():
        X_chain_1[col] = X_chain_1[col].fillna(X_train_1_v2[col].median())
df_chain['p_add_to_cart'] = model_1_v2.predict_proba(
    scaler_1_v2.transform(X_chain_1))[:, 1]

df_cart = df_chain[df_chain['added_to_cart'] == 1].copy()
X_chain_2 = df_cart[md2_features].copy()
for col in md2_features:
    if X_chain_2[col].isnull().any():
        X_chain_2[col] = X_chain_2[col].fillna(X_train_2[col].median())
df_cart['p_checkout'] = model_2.predict_proba(
    scaler_2.transform(X_chain_2))[:, 1]

df_checkout = df_cart[df_cart['started_checkout'] == 1].copy()
X_chain_3 = df_checkout[md3_features].copy()
for col in md3_features:
    if X_chain_3[col].isnull().any():
        X_chain_3[col] = X_chain_3[col].fillna(X_train_3[col].median())
df_checkout['p_purchase_given_checkout'] = model_3.predict_proba(
    scaler_3.transform(X_chain_3))[:, 1]

df_chain = df_chain.merge(
    df_cart[['unique_session', 'p_checkout']],
    on='unique_session', how='left'
)
df_chain = df_chain.merge(
    df_checkout[['unique_session', 'p_purchase_given_checkout']],
    on='unique_session', how='left'
)

df_chain['p_checkout']                = df_chain['p_checkout'].fillna(0)
df_chain['p_purchase_given_checkout'] = df_chain['p_purchase_given_checkout'].fillna(0)

df_chain['p_purchase_chain'] = (
    df_chain['p_add_to_cart'] *
    df_chain['p_checkout'] *
    df_chain['p_purchase_given_checkout']
)

print("Chain probability summary (test set, sessions with product views):")
print(df_chain[['p_add_to_cart', 'p_checkout',
                'p_purchase_given_checkout',
                'p_purchase_chain', 'purchased']].describe().round(4))

**Chain Rule Probability — Interpretation**

The table above summarises the distribution of predicted probabilities across
the three funnel stages for all 29,823 test sessions that included at least
one product view.

**Key observations:**

- **`p_add_to_cart` (mean: 0.340):** The model assigns a non-trivial average
  probability of cart addition to product-view sessions, but the distribution
  is wide (std: 0.234), reflecting genuine heterogeneity in user intent at
  the top of the funnel.

- **`p_checkout` (mean: 0.026) and `p_purchase_given_checkout` (mean: 0.009):**
  The 75th percentile of both variables is exactly 0, because the vast majority
  of sessions never reached the cart stage — so their checkout and purchase
  probabilities are set to 0 by construction in the chain rule. This is
  methodologically correct: a session that did not add to cart cannot progress
  to checkout.

- **`p_purchase_chain` (mean: 0.0051):** The average predicted joint probability
  of purchase across all product-view sessions is **0.51%**. The observed
  purchase rate in the test set is **0.83%** (`purchased` mean: 0.0083).
  The model slightly underestimates the overall conversion rate, which is
  expected given the conservative nature of multiplying three probabilities
  in sequence and the use of `class_weight='balanced'` at Stage 1. The order
  of magnitude is nonetheless correct, validating the coherence of the
  chain rule decomposition.

- **Maximum `p_purchase_chain`: 0.9944** — the model assigns near-certain
  purchase probability to a small subset of sessions, corresponding to users
  who combine very high engagement, identified login status, and prior purchase
  history across all three stages simultaneously.

### 3.6 Summary: AUC Comparison Across Funnel Stages

The table below summarises the performance of each model, along with the size of the test subset and the positive class rate at each stage. A higher AUC at a given stage indicates that the model is better able to distinguish between users who progress and those who drop off at that point.

In [ ]:
summary = pd.DataFrame([
    {
        'Stage': 'View → Add to Cart',
        'N (test)': len(y_test_1_v2),
        'Positive Rate': f"{y_test_1_v2.mean()*100:.1f}%",
        'ROC AUC': round(results_1_v2['auc'], 4)
    },
    {
        'Stage': 'Add to Cart → Checkout',
        'N (test)': len(y_test_2),
        'Positive Rate': f"{y_test_2.mean()*100:.1f}%",
        'ROC AUC': round(results_2['auc'], 4)
    },
    {
        'Stage': 'Checkout → Purchase',
        'N (test)': len(y_test_3),
        'Positive Rate': f"{y_test_3.mean()*100:.1f}%",
        'ROC AUC': round(results_3['auc'], 4)
    }
])
display(summary)


Three patterns are worth highlighting:

**1. The funnel narrows dramatically at the first transition.** Only 6.5%
of sessions with a product view result in a cart addition — this is the primary
bottleneck and the stage where the largest volume of potential customers is
lost. The 3,822 test sessions at Stage 1 shrink to 247 at Stage 2 and
87 at Stage 3.

**2. Discriminatory power decreases as users progress deeper into the funnel.**
AUC drops from 0.8395 at Stage 1 to 0.8159 at Stage 2 to 0.6704 at Stage 3.
This is structurally expected: users who reach later stages are increasingly
self-selected as high-intent, making them harder to distinguish from one
another using behavioral features alone.

**3. The low AUC at Stage 3 (0.6704) is itself an informative finding.**
It signals that the variables available in the GA4 dataset do not fully
explain checkout abandonment. The factors driving last-step dropout are
likely unobservable from session data — shipping costs revealed at checkout,
payment method friction, and technical issues in the payment flow. This
motivates the recommendation in Section 5 to instrument the checkout flow
with more granular GA4 events

### Summary Table: Key Feature Effects Across Funnel Stages

The table below summarises the direction and significance of each feature's effect
at each funnel stage. ↑ = positive and significant, ↓ = negative and significant,
~ = not significant, — = not applicable at that stage.

| Feature | M1: View→Cart | M2: Cart→Checkout | M3: Checkout→Purchase | Business Reading |
|---|---|---|---|---|
| `session_duration_sec` | ↑ | ↑ | ~ | Longer sessions signal committed browsers in early stages. Its effect becomes non-significant at purchase. |
| `ratio_engagement_duration` | ↑ | ~ | ~ | Active engagement drives early funnel progression; becomes non-significant later. |
| `user_id_binary` | ↑ | ↑ | ↑ | Identified users convert better at every stage. Login is a funnel-wide lever. |
| `logged_in_or_null_dummy` | ↑ | ↑ | ↑ | Identified users show higher intent across the funnel stages. |
| `log_avg_viewed_price` | ↓ | ~ | ~ | Price sensitivity is strongest at the viewing stage; effect diminishes deeper in the funnel. |
| `unique_items_viewed` | ↓ | ~ | ~ | Dispersive browsing hurts early progression. Effect becomes insignificant later. |
| `log_unique_categories_viewed` | ↑ | ↓ | ↓ | Initial category exploration helps cart addition; broad browsing becomes indecision later. |
| `items_viewed` | ↑ | ↓ | ~ | Viewing more items helps at the top of the funnel but can hurt at checkout. |
| `log_ratio_items_viewed_unique` | ↑ | ↓ | ↓ | Repeated views of same items drive cart addition but can hurt later. |
| `ratio_items_viewed>1` | ↑ | ↑ | ~ | Confirms pattern above in binary form. Effect becomes non-significant at purchase. |
| `log_time_view_to_cart` | ↓ | ↓ | ~ | Quicker add-to-cart decisions signal higher intent. Speed matters early. Effect becomes insignificant later. |
| `log_time_cart_to_checkout` | — | ~ | ↑ | Longer deliberation time between cart and checkout is positive for purchase. |
| `log_cart_value` | — | ~ | ~ | Higher cart value potentially positive for initiating checkout but effect becomes insignificant at purchase. |
| `log_checkout_events` | — | ~ | ~ | No significant effect. |
| `had_added_to_cart_before` | ↑ | ~ | ~ | Prior cart additions signal familiarity and intent, especially early in the funnel. Effect becomes insignificant later. |
| `n_added_to_cart_before` | ↑ | ~ | ~ | Prior cart additions signal familiarity and intent, especially early in the funnel. Becomes non-significant later. |
| `had_started_checkout_before` | — | ↑ | ~ | Previous checkout attempts signal warm, high-intent users. Effect becomes insignificant later. |
| `n_started_checkout_before` | — | ↑ | ~ | Repeated checkout starts is a strong positive signal for starting checkout but becomes non-significant at purchase. |
| `had_purchased_before` | — | — | ~ | Prior purchases are not significant for purchase probability. |
| `n_purchased_before` | — | — | ~ | Conditional on having purchased, higher counts may signal lower current-session urgency. Not significant. |
| `log_ga_session_number` | ↑ | ~ | ↑ | Higher session numbers can be positive at cart addition and purchase stages, but non-significant at checkout. |
| `time_of_day_segment_evening` | ↓ | ~ | ↓ | Evening sessions show systematically lower conversion at top of funnel and at purchase. |
| `time_of_day_segment_morning` | ~ | ~ | ~ | Morning has no significant effect relative to afternoon baseline. |
| `time_of_day_segment_night` | ~ | ~ | ↓ | Night shows a negative significant effect at purchase. |
| `num_sessions_per_user` | ~ | ~ | ↓ | Total sessions for a user is negative at purchase stage. |

### 3.7 Variable Impact on Final Purchase Probability (Chain Rule Perspective)

Understanding how each variable impacts the final purchase probability requires tracing its influence through each stage of the conversion funnel, as modeled by our three logistic regressions. A change in a variable at a given stage affects the probability of progression at that stage, which then cascades through the subsequent stages, ultimately influencing the overall likelihood of purchase.

Here's an interpretation of key variables and their compounded effects:


**1. `session_duration_sec`**
*   **Model 1 (View → Add to Cart):** Coefficient `0.0001`. A positive and significant effect, implying longer session durations increase the probability of adding to cart.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `0.0008`. Remains positive and significant, further supporting that longer sessions are associated with proceeding to checkout.
*   **Model 3 (Checkout → Purchase):** Coefficient `-1.864e-06`. Not significant, indicating its direct impact lessens at the final stage.
*   **Overall Impact:** This variable consistently predicts progression in the early and middle funnel stages, indicating that users spending more time on the site are more likely to initially engage, but its direct influence on the final purchase is not significant.

**2. `ratio_engagement_duration`**
*   **Model 1 (View → Add to Cart):** Coefficient `1.2979`. A positive and significant effect, suggesting that a higher proportion of session duration spent actively engaged increases the likelihood of adding to cart.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `0.2991`. Not significant, its effect is negligible at this stage.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.3776`. Not significant, implying its direct impact lessens at the final stage.
*   **Overall Impact:** Active engagement, especially in the earliest stage, drives users down the funnel, but its direct predictive power might be somewhat absorbed by other metrics like total time in later stages.

**3. `user_id_binary`**
*   **Model 1 (View → Add to Cart):** Coefficient `2.1477`. A strongly positive and significant effect. Users with an identified `user_id` (likely logged in or returning) are much more prone to adding items to their cart.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `1.4724`. Remains a strong positive predictor, indicating identified users are significantly more likely to proceed from cart to checkout.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.5627`. Continues to be positive and significant, showing that registered users have a higher probability of completing a purchase.
*   **Overall Impact:** Being a recognized user consistently boosts the probability of progression at every stage. Encouraging account creation or login has a profound positive multiplicative effect on final purchase probability.

**4. `logged_in_or_null_dummy`**
*   **Model 1 (View → Add to Cart):** Coefficient `1.6803`. A strongly positive and significant effect. Users who are logged in or whose login status is 'null' (compared to 'not logged') are more likely to add to cart.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `0.5849`. Positive and significant. Its effect remains positive at this stage.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.9079`. Positive and significant, indicating that users with this status are more likely to complete the purchase at checkout.
*   **Overall Impact:** This consistently positive effect suggests identified users (whether explicitly logged in or in the 'null' category for `login_status` which was treated as a distinct group) show higher conversion rates across the funnel.

**5. `unique_items_viewed`**
*   **Model 1 (View → Add to Cart):** Coefficient `-0.2601`. Negative and significant, suggesting that browsing a higher number of *unique* items decreases the probability of adding to cart (potentially indicating less focused browsing).
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.0177`. Not significant, its effect is negligible at this stage.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.1742`. Not significant.
*   **Overall Impact:** Broad initial exploration hinders early funnel progression, but its impact diminishes later on.

**6. `log_unique_categories_viewed`**
*   **Model 1 (View → Add to Cart):** Coefficient `1.3774`. Positive and significant, implying that viewing more unique categories (after log transformation) increases the likelihood of adding to cart.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-1.0429`. Negative and significant. This suggests that continuing to browse broadly across many categories *after* adding to cart is detrimental to proceeding to checkout.
*   **Model 3 (Checkout → Purchase):** Coefficient `-1.2179`. Negative and significant, indicating that broad category exploration is also negative for completing the purchase.
*   **Overall Impact:** There's a shift in effect: initial category exploration might lead to cart additions, but continued broad browsing becomes an indicator of indecision and decreases progression likelihood in later stages.

**7. `log_ratio_items_viewed_unique`**
*   **Model 1 (View → Add to Cart):** This variable was not included in Model 1.
*   **Model 2 (Add to Cart → Checkout):** This variable was not included in Model 2.
*   **Model 3 (Checkout → Purchase):** This variable was not included in Model 3.
*   **Overall Impact:** This variable was not directly included in the models but its binary version `ratio_items_viewed>1` was used.

**8. `ratio_items_viewed>1`**
*   **Model 1 (View → Add to Cart):** Coefficient `1.2214`. Positive and significant. This dummy variable (1 if ratio > 1) suggests repeated views increase add-to-cart probability.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `0.4656`. Positive and significant. Repeated views are beneficial for proceeding to checkout.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.0629`. Not significant.
*   **Overall Impact:** Repeatedly viewing the same items is beneficial for early funnel progression, but its impact on final purchase is not significant.

**9. `log_cart_value`**
*   **Model 1 (View → Add to Cart):** Not applicable (cart value is not available at this stage).
*   **Model 2 (Add to Cart → Checkout):** Coefficient `0.0174`. Not significant. A higher cart value might slightly encourage starting checkout.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.1225`. Not significant.
*   **Overall Impact:** The effect of cart value is not consistently significant across stages.

**10. `log_ratio_items_added_to_cart_unique`**
*   **Model 1 (View → Add to Cart):** Not applicable.
*   **Model 2 (Add to Cart → Checkout):** This variable was not included in Model 2.
*   **Model 3 (Checkout → Purchase):** This variable was not included in Model 3.
*   **Overall Impact:** This variable was not directly included in the models but its binary version `ratio_items_added_to_cart>1` was used.

**11. `ratio_items_added_to_cart>1`**
*   **Model 1 (View → Add to Cart):** Not applicable.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `0.4165`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.6629`. Not significant.
*   **Overall Impact:** Repeated additions to cart do not show a consistently significant impact on progression.

**12. `log_time_view_to_cart`**
*   **Model 1 (View → Add to Cart):** Coefficient `-0.4854`. Strongly negative and significant, indicating quicker decisions to add to cart are beneficial.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.1023`. Negative and significant. Longer time from view to cart is detrimental even at this stage.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.0226`. Not significant.
*   **Overall Impact:** Quick decision-making in the early stages of the funnel is a strong positive indicator for progression.

**13. `log_time_cart_to_checkout`**
*   **Model 1 (View → Add to Cart):** Not applicable.
*   **Model 2 (Add to Cart → Checkout):** This variable was not included in Model 2.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.3266`. Positive and highly significant. A longer time spent between cart and checkout is beneficial for completing the purchase, possibly indicating a thoughtful, deliberate buying process.
*   **Overall Impact:** This variable exhibits a nuanced positive effect at the final stage, suggesting that a considered approach rather than hasty decisions after adding to cart is linked to higher conversion.

**14. `time_of_day_segment_evening`**
*   **Model 1 (View → Add to Cart):** Coefficient `-0.1668`. Negative and significant, suggesting a lower add-to-cart probability in the evening compared to the reference category (afternoon).
*   **Model 2 (Add to Cart → Checkout):** Coefficient `0.1096`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.6674`. Negative and significant, implying lower purchase probability in the evening.
*   **Overall Impact:** Evening times tend to be less favorable for conversion at various stages of the funnel.

**15. `time_of_day_segment_morning`**
*   **Model 1 (View → Add to Cart):** Coefficient `0.0219`. Not significant.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.0181`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.3393`. Not significant.
*   **Overall Impact:** Morning times generally show a slightly negative or neutral impact across stages compared to the reference (afternoon).

**16. `time_of_day_segment_night`**
*   **Model 1 (View → Add to Cart):** Coefficient `-0.0130`. Not significant.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.3846`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `-1.1392`. Negative and significant.
*   **Overall Impact:** Night times show a negative impact at the final purchase stage.

**17. `had_viewed_product_before`**
*   **Model 1 (View → Add to Cart):** This variable was not included in Model 1.
*   **Model 2 (Add to Cart → Checkout):** This variable was not included in Model 2.
*   **Model 3 (Checkout → Purchase):** This variable was not included in Model 3.
*   **Overall Impact:** This variable was not directly included in the models but its count version `n_viewed_product_before` was used.

**18. `n_viewed_product_before`**
*   **Model 1 (View → Add to Cart):** Coefficient `-0.0784`. Negative and significant, suggesting that more previous product views (without conversion) might indicate indecision.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.0930`. Negative and significant, suggesting continued indecision.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.0896`. Not significant.
*   **Overall Impact:** More previous product views can be a negative signal early on, but its impact diminishes later in the funnel.

**19. `had_added_to_cart_before`**
*   **Model 1 (View → Add to Cart):** This variable was not included in Model 1.
*   **Model 2 (Add to Cart → Checkout):** This variable was not included in Model 2.
*   **Model 3 (Checkout → Purchase):** This variable was not included in Model 3.
*   **Overall Impact:** This variable was not directly included in the models but its count version `n_added_to_cart_before` was used.

**20. `n_added_to_cart_before`**
*   **Model 1 (View → Add to Cart):** Coefficient `0.3346`. Positive and significant. More previous cart additions (without conversion) increase the current likelihood of adding to cart.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.2397`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.6039`. Not significant.
*   **Overall Impact:** A history of cart additions is positive for early progression, but its impact diminishes later.

**21. `had_started_checkout_before`**
*   **Model 1 (View → Add to Cart):** Not applicable.
*   **Model 2 (Add to Cart → Checkout):** This variable was not included in Model 2.
*   **Model 3 (Checkout → Purchase):** This variable was not included in Model 3.
*   **Overall Impact:** This variable was not directly included in the models but its count version `n_started_checkout_before` was used.

**22. `n_started_checkout_before`**
*   **Model 1 (View → Add to Cart):** Not applicable.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `1.0129`. Strongly positive and significant. More previous checkout starts increase the current likelihood of starting checkout.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.1499`. Not significant.
*   **Overall Impact:** A history of initiating checkout is a strong positive indicator for progression at the middle stage, but its impact on actual purchase is not clear.

**23. `had_purchased_before`**
*   **Model 1 (View → Add to Cart):** Not applicable.
*   **Model 2 (Add to Cart → Checkout):** Not applicable.
*   **Model 3 (Checkout → Purchase):** This variable was not included in Model 3.
*   **Overall Impact:** This variable was not directly included in the models but its count version `n_purchased_before` was used.

**24. `n_purchased_before`**
- **Model 3 (Checkout → Purchase):** Coefficient `1.1199`. Not significant.
  This result suggests that having purchased multiple times before does not significantly influence the current session's purchase probability.

**25. `log_ga_session_number`**
*   **Model 1 (View → Add to Cart):** Coefficient `0.0814`. Positive and significant. A higher session number is associated with increased add-to-cart probability.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.0790`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.3036`. Positive and significant. Users in later GA sessions are more likely to complete a purchase.
*   **Overall Impact:** Shows a mixed impact, being positive at the initial and final stages, but not significant in the middle of the funnel.

**26. `items_viewed`**
*   **Model 1 (View → Add to Cart):** Coefficient `0.1693`. Positive and significant. A higher number of items viewed slightly increases the probability of adding to cart.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.1035`. Negative and significant. More items viewed decrease the probability of proceeding to checkout.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.0273`. Not significant.
*   **Overall Impact:** Initial viewing activity can be positive, but excessive viewing may indicate indecision or be detrimental later in the funnel.

**27. `items_added_to_cart`**
*   **Model 1 (View → Add to Cart):** Not applicable.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.0610`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `0.5525`. Not significant.
*   **Overall Impact:** The total number of items added to cart does not show a consistently significant impact on progression to checkout or purchase.

**28. `log_avg_viewed_price`**
*   **Model 1 (View → Add to Cart):** Coefficient `-0.5411`. Strongly negative and significant. Viewing higher-priced items decreases the probability of adding to cart.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `-0.1511`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.3284`. Not significant.
*   **Overall Impact:** Price sensitivity appears to be a factor at the initial stage of adding to cart.

**29. `log_checkout_events`**
*   **Model 1 (View → Add to Cart):** Not applicable.
*   **Model 2 (Add to Cart → Checkout):** Not applicable.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.3718`. Not significant.
*   **Overall Impact:** This variable does not show a significant impact on actual purchase.

**30. `num_sessions_per_user`**
*   **Model 1 (View → Add to Cart):** Coefficient `0.0042`. Not significant.
*   **Model 2 (Add to Cart → Checkout):** Coefficient `0.0057`. Not significant.
*   **Model 3 (Checkout → Purchase):** Coefficient `-0.0315`. Negative and significant.
*   **Overall Impact:** The total number of sessions a user has had negatively impacts conversion at the final stage.

## 4 Funnel-Based Random Forest (Chain Rule Decomposition)

We now replicate the same three-stage funnel structure using a **Random Forest classifier**. The decomposition, subsetting logic, and user-aware train/test split are identical to Section 3, allowing direct comparison of performance between the two approaches.

Random Forest differs from Logistic Regression in three key ways relevant to this analysis:
- It captures **non-linear relationships** and **interaction effects** between features without requiring explicit specification
- Feature importance is derived from **mean decrease in impurity** across all trees, rather than from linear coefficients
- It does **not require feature scaling**, so `StandardScaler` is omitted

To assess the stability of each model's performance, we add **5-fold Group Cross-Validation** using `GroupKFold`, which ensures that all sessions belonging to the same user are kept within the same fold — consistent with the panel-aware split used throughout this analysis.





### 4.1 Imports and Hyperparameter Tuning Setup

Before fitting the three models, we perform a lightweight hyperparameter search using `RandomizedSearchCV` with `GroupKFold` to select the best combination of `n_estimators`, `max_depth`, and `min_samples_leaf`. The search is run on the training set only, and the best parameters are then used to fit the final model evaluated on the held-out test set.

In [ ]:
pd.set_option('mode.chained_assignment', None)

# Hyperparameter search space
param_dist = {
    'n_estimators':    randint(100, 400),
    'max_depth':       [None, 5, 10, 15, 20],
    'min_samples_leaf': randint(5, 50),
    'max_features':    ['sqrt', 'log2']
}

N_ITER   = 7   # number of random combinations to try
N_SPLITS = 5    # number of GroupKFold folds
RANDOM_STATE = 42

### 4.2 Helper Function: Fit, Tune, and Evaluate a Random Forest Model

The function below:
1. Runs a `RandomizedSearchCV` with `GroupKFold` on the training set to find the best hyperparameters
2. Refits the best model on the full training set
3. Evaluates on the held-out test set (ROC AUC, classification report)
4. Runs a 5-fold `GroupKFold` cross-validation on the training set to report mean AUC ± std
5. Plots the ROC curve and top 15 features by importance

In [ ]:
def fit_evaluate_rf(X_train, X_test, y_train, y_test,
                    groups_train, model_name):
    """
    Tunes and fits a Random Forest classifier using GroupKFold CV,
    evaluates on the test set, and plots ROC curve + feature importance.

    Parameters
    ----------
    X_train, X_test  : feature DataFrames
    y_train, y_test  : binary target Series
    groups_train     : array of user_pseudo_id for each row in X_train
                       (used to keep users intact across CV folds)
    model_name       : string label for plots and reports

    Returns
    -------
    best_model : fitted RandomForestClassifier with best hyperparameters
    results    : dict with key metrics
    """

    gkf = GroupKFold(n_splits=N_SPLITS)

    # --- Hyperparameter search ---
    base_rf = RandomForestClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    search = RandomizedSearchCV(
        estimator=base_rf,
        param_distributions=param_dist,
        n_iter=N_ITER,
        scoring='roc_auc',
        cv=gkf,
        refit=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0
    )
    search.fit(X_train, y_train, groups=groups_train)

    best_model  = search.best_estimator_
    best_params = search.best_params_

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  Best hyperparameters: {best_params}")

    # --- Test set evaluation ---
    y_pred       = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_pred_proba)

    print(f"\n  Positive class rate (test): {y_test.mean()*100:.2f}%")
    print(f"  ROC AUC Score (test):       {auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # --- GroupKFold CV on training set ---
    cv_scores = cross_val_score(
        best_model, X_train, y_train,
        cv=gkf,
        groups=groups_train,
        scoring='roc_auc',
        n_jobs=-1
    )
    print(f"  Cross-Validation AUC (train, {N_SPLITS}-fold GroupKFold):")
    print(f"  Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")
    print(f"  Per-fold: {np.round(cv_scores, 4)}")

    # --- ROC Curve ---
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(fpr, tpr, color='darkorange', lw=2,
                 label=f'AUC = {auc:.3f}')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'ROC Curve — {model_name}')
    axes[0].legend()

    # --- Feature Importance ---
    imp_df = pd.DataFrame({
        'feature':    X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)

    axes[1].barh(imp_df['feature'], imp_df['importance'],
                 color='darkorange', edgecolor='white')
    axes[1].set_xlabel('Mean Decrease in Impurity')
    axes[1].set_title(f'Top 15 Features — {model_name}')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    return best_model, {
        'model_name':    model_name,
        'auc':           auc,
        'cv_scores':     cv_scores,
        'imp_df':        imp_df,
        'best_params':   best_params,
        'y_pred_proba':  y_pred_proba,
        'y_test':        y_test
    }

### 4.3 Model 1: P(Add to Cart | Viewed Product)

Same subset and feature set as Logistic Regression Model 1: sessions where `viewed_product == 1`, predicting `added_to_cart`. Only features observable at the viewing stage are included.

In [ ]:
# Subset: sessions with at least one product view
df_m1_rf = df_base[df_base['viewed_product'] == 1].copy()

print(f"[RF Model 1] Sessions with product views: {len(df_m1_rf)}")
print(f"[RF Model 1] Positive rate (added to cart): {df_m1_rf['added_to_cart'].mean()*100:.2f}%")

# Use md1_features_v2 (time_of_day_segment, no hour or date dummies)
# for consistency with the final LR specification (Model 1.2)
X_train_rf1 = df_m1_rf.loc[df_m1_rf['user_pseudo_id'].isin(users_train), md1_features_v2].copy()
X_test_rf1  = df_m1_rf.loc[df_m1_rf['user_pseudo_id'].isin(users_test),  md1_features_v2].copy()
y_train_rf1 = df_m1_rf.loc[df_m1_rf['user_pseudo_id'].isin(users_train), 'added_to_cart']
y_test_rf1  = df_m1_rf.loc[df_m1_rf['user_pseudo_id'].isin(users_test),  'added_to_cart']

groups_rf1 = df_m1_rf.loc[df_m1_rf['user_pseudo_id'].isin(users_train),
                            'user_pseudo_id'].values

for col in md1_features_v2:
    if X_train_rf1[col].isnull().any():
        med = X_train_rf1[col].median()
        X_train_rf1[col] = X_train_rf1[col].fillna(med)
        X_test_rf1[col]  = X_test_rf1[col].fillna(med)

rf_model_1, rf_results_1 = fit_evaluate_rf(
    X_train_rf1, X_test_rf1, y_train_rf1, y_test_rf1,
    groups_train=groups_rf1,
    model_name="RF Model 1: P(Add to Cart | Viewed Product)"
)

### 4.4 Model 2: P(Start Checkout | Added to Cart)

Same subset and feature set as Logistic Regression Model 2: sessions where `added_to_cart == 1`, predicting `started_checkout`. Cart-specific features are now included.

In [ ]:
# Subset: sessions with at least one cart addition
df_m2_rf = df_base[df_base['added_to_cart'] == 1].copy()

print(f"[RF Model 2] Sessions with cart additions: {len(df_m2_rf)}")
print(f"[RF Model 2] Positive rate (started checkout): {df_m2_rf['started_checkout'].mean()*100:.2f}%")

X_train_rf2 = df_m2_rf.loc[df_m2_rf['user_pseudo_id'].isin(users_train), md2_features].copy()
X_test_rf2  = df_m2_rf.loc[df_m2_rf['user_pseudo_id'].isin(users_test),  md2_features].copy()
y_train_rf2 = df_m2_rf.loc[df_m2_rf['user_pseudo_id'].isin(users_train), 'started_checkout']
y_test_rf2  = df_m2_rf.loc[df_m2_rf['user_pseudo_id'].isin(users_test),  'started_checkout']

groups_rf2 = df_m2_rf.loc[df_m2_rf['user_pseudo_id'].isin(users_train),
                            'user_pseudo_id'].values

for col in md2_features:
    if X_train_rf2[col].isnull().any():
        med = X_train_rf2[col].median()
        X_train_rf2[col] = X_train_rf2[col].fillna(med)
        X_test_rf2[col]  = X_test_rf2[col].fillna(med)

rf_model_2, rf_results_2 = fit_evaluate_rf(
    X_train_rf2, X_test_rf2, y_train_rf2, y_test_rf2,
    groups_train=groups_rf2,
    model_name="RF Model 2: P(Start Checkout | Added to Cart)"
)

### 4.5 Model 3: P(Purchase | Started Checkout)

Same subset and feature set as Logistic Regression Model 3: sessions where `started_checkout == 1`, predicting `purchased`. Checkout-specific features are included. `time_checkout_to_purchase` remains excluded as it constitutes direct target leakage.

In [ ]:
# Subset: sessions where checkout was started
df_m3_rf = df_base[df_base['started_checkout'] == 1].copy()

print(f"[RF Model 3] Sessions with checkout started: {len(df_m3_rf)}")
print(f"[RF Model 3] Positive rate (purchased): {df_m3_rf['purchased'].mean()*100:.2f}%")

X_train_rf3 = df_m3_rf.loc[df_m3_rf['user_pseudo_id'].isin(users_train), md3_features].copy()
X_test_rf3  = df_m3_rf.loc[df_m3_rf['user_pseudo_id'].isin(users_test),  md3_features].copy()
y_train_rf3 = df_m3_rf.loc[df_m3_rf['user_pseudo_id'].isin(users_train), 'purchased']
y_test_rf3  = df_m3_rf.loc[df_m3_rf['user_pseudo_id'].isin(users_test),  'purchased']

groups_rf3 = df_m3_rf.loc[df_m3_rf['user_pseudo_id'].isin(users_train),
                            'user_pseudo_id'].values

for col in md3_features:
    if X_train_rf3[col].isnull().any():
        med = X_train_rf3[col].median()
        X_train_rf3[col] = X_train_rf3[col].fillna(med)
        X_test_rf3[col]  = X_test_rf3[col].fillna(med)

rf_model_3, rf_results_3 = fit_evaluate_rf(
    X_train_rf3, X_test_rf3, y_train_rf3, y_test_rf3,
    groups_train=groups_rf3,
    model_name="RF Model 3: P(Purchase | Started Checkout)"
)

### 4.6 Final Comparison: Logistic Regression vs. Random Forest

We conclude by comparing the test-set AUC and cross-validation AUC of both approaches across all three funnel stages. A large gap between CV AUC and test AUC would signal overfitting; similar values indicate a stable and well-generalizing model.

In [ ]:
comparison = pd.DataFrame([
    {
        'Stage':        'View → Add to Cart',
        'LR AUC':       round(results_1_v2['auc'], 4),
        'RF AUC (test)': round(rf_results_1['auc'], 4),
        'RF CV AUC (mean)': round(rf_results_1['cv_scores'].mean(), 4),
        'RF CV AUC (std)':  round(rf_results_1['cv_scores'].std(), 4),
    },
    {
        'Stage':        'Add to Cart → Checkout',
        'LR AUC':       round(results_2['auc'], 4),
        'RF AUC (test)': round(rf_results_2['auc'], 4),
        'RF CV AUC (mean)': round(rf_results_2['cv_scores'].mean(), 4),
        'RF CV AUC (std)':  round(rf_results_2['cv_scores'].std(), 4),
    },
    {
        'Stage':        'Checkout → Purchase',
        'LR AUC':       round(results_3['auc'], 4),
        'RF AUC (test)': round(rf_results_3['auc'], 4),
        'RF CV AUC (mean)': round(rf_results_3['cv_scores'].mean(), 4),
        'RF CV AUC (std)':  round(rf_results_3['cv_scores'].std(), 4),
    },
])

print("\n" + "="*70)
print("  LOGISTIC REGRESSION vs. RANDOM FOREST — FUNNEL COMPARISON")
print("="*70)
display(comparison)

### 4.7 Results Summary and Business Interpretation

### Overall Comparison: Logistic Regression vs. Random Forest

| Stage | LR AUC | RF AUC (test) | RF CV AUC (mean ± std) |
|---|---|---|---|
| View → Add to Cart | 0.8243 | 0.8533 | 0.8664 ± 0.0082 |
| Add to Cart → Checkout | 0.7863 | 0.8515 | 0.8211 ± 0.0208 |
| Checkout → Purchase | 0.7034 | 0.7335 | 0.7302 ± 0.0270 |

The Random Forest generally outperforms Logistic Regression at every funnel stage, with the largest relative improvement observed at the 'Checkout → Purchase' stage. Three observations are worth highlighting beyond the raw AUC numbers:

**Generalization is strong at Stages 1 and 2.** The CV AUC and test AUC are closely aligned for RF Models 1 and 2 (gaps of 0.0131 and 0.0304 respectively), confirming that both models generalize well and are not overfitting to the training partition.

**Stage 3 shows a relatively stable CV-test alignment (0.7302 vs. 0.7335).** The mean CV AUC is very close to the test AUC, with a standard deviation of 0.0270, indicating a stable performance despite the smaller sample size at this stage.

**The largest relative improvement in AUC is at Stage 3 (+3.01 percentage points over LR).** This confirms that the relationship between pre-checkout behavioral features and the final purchase decision is genuinely non-linear — interactions between features that LR cannot capture are exploited effectively by the Random Forest.

---

### RF Model 1 — P(Add to Cart | Viewed Product) — AUC: 0.8533

The RF improves over LR Model 1 by **+2.9 AUC points** using the same feature set (`md1_features_v2`, with `time_of_day_segment` and no granular hour or date dummies). Cross-validation is stable across all five folds (range: 0.8536–0.8787, std: 0.0082), confirming reliable generalization.

The feature importance plot clarifies the hierarchy of predictors:

- `session_duration_sec` and `ratio_engagement_duration` are the dominant features, collectively accounting for a substantial portion of predictive power at this stage. The RF exploits non-linear thresholds in these variables, which the LR could only approximate linearly.
- `log_avg_viewed_price` ranks highly, indicating that the RF captures non-linear price effects; very high-priced item sessions behave differently from moderately priced ones in ways a linear coefficient cannot express.
- `items_viewed`, `log_unique_categories_viewed`, and `ratio_items_viewed>1` confirm that browsing depth, category exploration, and focus are meaningful signals.
- `user_id_binary` and `logged_in_or_null_dummy` remain important, suggesting that identified users have a higher propensity to add to cart.
- `time_of_day_segment` dummies appear at the bottom of the top 15 with minimal importance, consistent with the LR finding that only the evening segment was marginally significant.

---

### RF Model 2 — P(Start Checkout | Added to Cart) — AUC: 0.8515

_Note: `total_engagement_time` was excluded from this model's feature set as well._

The RF improves over LR Model 2 by **+6.52 AUC points**. Cross-validation is stable (mean: 0.8211, std: 0.0208, per-fold range: 0.7869–0.8486).

The feature importance profile at this stage differs meaningfully from Stage 1:

- `session_duration_sec` remains a top predictor, confirming the continued importance of overall session length.
- `ratio_engagement_duration` also retains significant importance, highlighting the value of active engagement as users move from cart to checkout.
- `user_id_binary` and `logged_in_or_null_dummy` rise in importance, consistent with the LR finding that the login effect becomes increasingly important as users approach purchase.
- `log_cart_value` and `log_time_view_to_cart` rank highly; users with higher cart values and quicker add-to-cart decisions are more likely to proceed to checkout.
- History features like `n_started_checkout_before` also appear, confirming the longitudinal signal identified in the LR.

---

### RF Model 3 — P(Purchase | Started Checkout) — AUC: 0.7335

_Note: `total_engagement_time` was excluded from this model's feature set._

The RF delivers a notable improvement at this stage (+3.01 AUC points over LR), confirming genuine non-linearity in the checkout-to-purchase transition. The CV AUC (0.7302 ± 0.0270) is very close to the test AUC (0.7335), indicating robust generalization despite the small sample size (269 test sessions).

The feature importance profile at Stage 3 is markedly different from the previous two stages:

- `session_duration_sec` remains the top predictor — at the checkout stage, total time on site continues to be a strong indicator.
- `ratio_engagement_duration` ranks second, but its effect here, as in the LR, can be nuanced, showing how engagement intensity plays out at the final stage.
- `log_cart_value` and `log_time_cart_to_checkout` both rank highly. A higher cart value and longer deliberation time between cart and checkout are key predictors, suggesting that thoughtful consideration might lead to conversion at this final step.
- `user_id_binary` and `logged_in_or_null_dummy` retain importance, emphasizing the value of recognized users even at the point of purchase.
- `log_checkout_events` and `log_avg_viewed_price` are also significant, with `log_checkout_events` possibly capturing complexities in the checkout process, and `log_avg_viewed_price` reflecting continued price sensitivity.
- `num_sessions_per_user` shows up, indicating the impact of overall user history on final purchase likelihood.

## 5 Business Recommendations for Versace — Executive Summary

The following recommendations synthesise the findings from the Logistic
Regression (Section 3) and Random Forest (Section 4) models. They are ordered
by estimated business impact, from the highest-volume opportunity to the most
targeted intervention.

---

### Recommendation 1 — Invest in Content Quality to Increase Time on Site
**Evidence:** `total_engagement_time` is the single strongest predictor of
funnel progression at every stage in both models. In LR Model 1.2, its
coefficient is 1.591e-06 per millisecond — meaning a user with 200,000ms
of engagement (approximately 3.3 minutes) has a log-odds of cart addition
that is 0.32 higher than a user with no engagement, all else equal. In the
Random Forest, it accounts for the largest share of feature importance at
Stages 1 and 2 by a wide margin.

**Action:** Invest in richer on-site content — editorial storytelling, video
lookbooks, 360° product views, and styling guides consistent with Versace's
brand identity. These formats increase active engagement time and have a
multiplicative positive effect on conversion simultaneously across all three
funnel stages.

---

### Recommendation 2 — Implement Progressive Login Prompts
**Evidence:** `user_id_binary` is a strong positive predictor at all three
stages (LR coefficients: +1.847, +1.698, +0.550). To quantify the effect
at Stage 1: the odds of cart addition for an identified user are e^1.847 ≈
**6.3x higher** than for an unidentified user, all else equal. The effect
remains substantial at checkout (`logged_in_or_null_dummy`: +0.930, odds
ratio ≈ 2.5x).

**Action:** Introduce progressive login prompts at the cart stage rather
than forcing account creation at checkout. Communicate concrete benefits
of registration — order tracking, saved preferences, exclusive early access.
Even a modest increase in the share of identified users would compound
positively across all three funnel stages simultaneously.

---

### Recommendation 3 — Invest in Guided Navigation to Reduce Dispersive Browsing
**Evidence:** `log_unique_categories_viewed` is negative and significant at
Stages 2 and 3 (LR coefficients: -1.385 and -0.718). `unique_items_viewed`
is negative at Stages 1 and 2. Users who browse broadly across many
categories after adding to cart are systematically less likely to proceed
to checkout or purchase.

**Action:** Invest in curated navigation tools — editorial "complete the
look" pages, AI-powered recommendation engines, and category-specific
landing pages — that keep user attention focused once they have engaged
with a product. For users who have already added items to their cart,
in-session nudges within the cart view could prevent further aimless
browsing.

---

### Recommendation 4 — Deploy Real-Time Nudges for Cart-to-Checkout Stalling
**Evidence:** `log_time_cart_to_checkout` is positively significant in LR
Model 3 (coef: +0.224) and ranks as the fourth most important feature in
RF Model 3. However, this positive effect reflects deliberate consideration
— not hesitation. The opportunity lies in users who stall excessively: the
RF captures a non-linear threshold beyond which longer cart-to-checkout
times become a negative signal.

**Action:** Set a defined inactivity threshold (e.g. 30–60 minutes after
cart addition without checkout initiation) and trigger a real-time nudge
— a browser notification, on-site banner, or follow-up email — offering
a time-limited incentive such as complimentary shipping. This directly
targets the hesitation window identified by the model.

---

### Recommendation 5 — Display Full Cost Breakdown Earlier in the Funnel
**Evidence:** `log_cart_value` (LR Model 3 coef: -0.408) and
`log_avg_viewed_price` are both important in RF Model 3, confirming that
price sensitivity is a key driver of last-step abandonment. The low AUC
at Stage 3 (LR: 0.702, RF: 0.782) suggests that factors revealed only
at checkout — shipping costs, duties, final total — account for a
substantial portion of unexplained abandonment.

**Action:** Display the complete cost breakdown — product price, estimated
shipping, applicable duties for international users — on the product page
and cart view, before the user reaches the checkout. Removing the price
surprise at the final step is one of the highest-ROI interventions
available for checkout abandonment reduction.

---

### Recommendation 6 — Re-engage Evening Browsers in Afternoon Windows
**Evidence:** `time_of_day_segment_evening` is the only time segment that
is statistically significant across multiple stages: negative in LR Model
1.2 (coef: -0.119, p=0.013) and LR Model 3 (coef: -0.356, p=0.044).
Evening sessions are systematically less likely to result in cart additions
or completed purchases compared to the afternoon baseline.

**Action:** For users who browse in the evening without converting, deploy
retargeting campaigns — paid social, email — scheduled to reach them
during afternoon hours (12:00–17:00), when the model predicts higher
conversion probability. This is a low-cost, immediately actionable lever
that requires no changes to the website itself.

---

### Recommendation 7 — Instrument the Checkout Flow with Granular GA4 Events
**Evidence:** The AUC at Stage 3 (LR: 0.702, RF: 0.782) is the lowest
across all models, and `log_checkout_events` is the strongest negative
predictor (LR coef: -0.820). The current GA4 data captures the number
of checkout interactions but not their nature — whether a user
encountered a payment error, hesitated on the shipping cost page, or
abandoned after seeing the final total.

**Action:** Instrument the checkout flow with more granular GA4 custom
events: time spent on the payment page, payment method selected,
validation errors encountered, and the specific step at which abandonment
occurred. This richer event stream would allow future models to identify
the exact friction points within checkout, closing the explanatory gap
that behavioral session data alone cannot fill.